# Vocabulary Preprocessing Pipeline

**Input**: CSV with columns: `target_word`, `duolingo_translations`  

Goal: Generate all possible translation forms to expand the possible mappings of the target word. More translations mean more substitutions and more exposure to target word. BUT the translations and forms have to be morphologically, linguistically accurate (to the best of computational abilities).

In [ ]:
# TODO:
# investigate 'novel' > generates [powiedzenia, powiedzenie, powiedzeniem, powiedzeniu]
# NEW LANGUAGE PAIR: 
# - change lemminflect 
# - change morfeusz2
# - change diki dictionary scraping 
# - get new morfological mapping table for generation
# - get new word frequency lists

#URGENT:
# remove "jest" from ranslations
# czym są , są, to,
# in general remove polish stop words from translations, currently only done to targets

In [ ]:
import morfeusz2
import pandas as pd
import logging
from collections import defaultdict, Counter
from pathlib import Path
from lemminflect import getAllLemmas, getInflection

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

morf = morfeusz2.Morfeusz(praet='composite')

# INPUT_CSV = Path("C:/Users/rrkar/Downloads/duolingo_vocab_tata_16052026_2026-05-19_2130_.csv")
INPUT_CSV = Path("C:/Users/rrkar/Downloads/duolingo_vocab_mama_16052026_2026-05-19_2131.csv")

## NEW METHOD

### Stage 0 - Load data

In [ ]:
df_raw = pd.read_csv(INPUT_CSV, sep=";", dtype=str)
df_raw.columns = [c.strip() for c in df_raw.columns]
assert "target" in df_raw.columns
assert "duolingo_translations" in df_raw.columns

df_raw["target"] = df_raw["target"].str.strip()
df_raw["duolingo_translations"] = df_raw["duolingo_translations"].str.strip()
print(f"Loaded {len(df_raw)} rows.")

#### investigation of capitalisations and punctuation

In [ ]:
import pandas as pd
tmp=df_raw.copy()
# phrase vs single word in target
tmp["is_phrase"] = tmp["target"].str.contains(" ")

# split translations
tmp["tokens"] = tmp["duolingo_translations"].str.split(",")

# clean tokens
tmp["tokens"] = tmp["tokens"].apply(lambda x: [t.strip() for t in x])

# one token per row
tmp = tmp.explode("tokens")

# capitalization check (first letter uppercase)
tmp["is_capitalised"] = tmp["tokens"].str.match(r"^[A-Z]")

tmp[(tmp["is_phrase"] == False) & (tmp["is_capitalised"] == True)]


In [ ]:
tmp[(tmp["target"].str.contains(r"[^A-Za-z]")) & (tmp['is_phrase']==False)].head(60)

#### Stage 0 - deduplication

In [ ]:
import re

words = [
    # "a", 
    # "an",
    # "the",
    "to",
    # 'and',
    # 'at'
    # "of",
    # 'for',
    #   'that'
    # 'after',
    # 'go',
    # 'in'
    # 'in a',
    # 'in the',
    # "on"
    ]


pattern = r"^(" + "|".join(map(re.escape, words)) + r")\b"
starts_with = re.compile(pattern, flags=re.IGNORECASE)
start_matches = df_raw[df_raw["target"].str.match(starts_with, na=False)]

# for target in start_matches["target"]:
    # print(target)
start_matches[["target",'duolingo_translations']]

In [ ]:
import re

stopwords = set(words)

# Escape in case a stopword contains regex characters
pattern = r"\b(" + "|".join(map(re.escape, sorted(stopwords))) + r")\b\s*$"
ends_with = re.compile(pattern, flags=re.IGNORECASE)

end_matches = df_raw[df_raw["target"].str.contains(ends_with, na=False)]

# for target in end_matches["target"]:
#     print(target)
end_matches[["target",'duolingo_translations']]

In [ ]:
def strip_target_and_translations(target: str, translations: str):
    target_tokens = target.split()
    translations = [t.strip() for t in translations.split(",")]

    def clean_translations(remove_start=None, remove_end=None):
        cleaned = []

        for t in translations:
            toks = t.split()

            if remove_start:
                while toks and toks[0].lower() in remove_start:
                    toks = toks[1:]

            if remove_end:
                while toks and toks[-1].lower() in remove_end:
                    toks = toks[:-1]

            cleaned.append(" ".join(toks).strip())

        return cleaned

    # 1. leading a/an/and/after  -> PO (lead rule)
    if target_tokens and target_tokens[0].lower() in {"a", "an", "and", "after"}:
        target_tokens = target_tokens[1:]
        translations = clean_translations(remove_start={"po"})

    # 2. trailing to/at/the/and
    if target_tokens and target_tokens[-1].lower() in {"to", "at", "the", "and"}:
        target_tokens = target_tokens[:-1]
        translations = clean_translations(remove_end={"do"})

    # 3. leading single-token article-like collapse
    if len(target_tokens) == 2 and target_tokens[0].lower() in {"the", "in"}:
        target_tokens = target_tokens[1:]
        translations = clean_translations(remove_start={"w", "z", "za", "na"})

    # 4. leading "in the", "in a"
    if len(target_tokens) == 3 and target_tokens[0].lower() == "in" and target_tokens[1].lower() in {"a","the"}:
        target_tokens = target_tokens[2:]
        translations = clean_translations(remove_start={"w", "z", "za", "na"})

    # 5. trailing of/for/that
    if len(target_tokens) == 2 and target_tokens[-1].lower() in {"of", "for", "that"}:
        target_tokens = target_tokens[:-1]
        translations = clean_translations(remove_end={"z", "za", "na", "dla"})

    target_out = " ".join(target_tokens).strip()
    translations_out = ", ".join(t for t in translations if t)

    return target_out, translations_out

# For target stripping stats at the end
raw_targets_before = set(df_raw["target"])

df_raw['target_raw']=df_raw['target'].copy()
df_raw['duolingo_translations_raw']=df_raw['duolingo_translations'].copy()
df_raw[["target", "duolingo_translations"]] = (
    df_raw.apply(lambda row: strip_target_and_translations(row["target"], row["duolingo_translations"]),
        axis=1,
        result_type="expand",
    )
)

In [ ]:
# def strip_target_phrase(phrase: str) -> str:
#     tokens = phrase.split()

#     # leading "a"/"an" # lead 'po'
#     if tokens and tokens[0].lower() in {"a", "an", "and", "after"}:
#         tokens = tokens[1:]

#     # trailing "to" — always safe per your rule (no length condition specified)
#     if tokens and tokens[-1].lower() in {"to", "at", "the", "and"}:
#         tokens = tokens[:-1]
    
#     # leading "the" only if it collapses phrase to exactly 1 token  # trail w,z,za,na
#     if len(tokens) == 2 and tokens[0].lower() in {"the","in"}:
#         tokens = tokens[1:]

#     # leading "in the" only if it collapses phrase to exactly 1 token # trail w,z,za,na
#     if len(tokens) == 3 and tokens[0].lower() == "in" and tokens[1].lower() == "the":
#         tokens = tokens[2:]

#     # leading "in a" only if it collapses phrase to exactly 1 token # trail w,z,za,na
#     if len(tokens) == 3 and tokens[0].lower() == "in" and tokens[1].lower() == "the":
#         tokens = tokens[2:]

#     # trailing "of"/"for"/"that" only if it collapses phrase to exactly 1 token # trail z,za,na,dla
#     if len(tokens) == 2 and tokens[-1].lower() in {"of", "for", "that"}:
#         tokens = tokens[:-1]

#     # final normalization
#     cleaned = " ".join(tokens).strip()
    
#     return cleaned

# df_raw['target_raw']=df_raw['target'].copy()
# df_raw['duolingo_translations_raw']=df_raw['duolingo_translations'].copy()
# df_raw["target"] = df_raw["target"].apply(strip_target_phrase)
# df_raw = df_raw[df_raw["target"].str.len() > 1]  # drop single-letter targets emptied out


In [ ]:
# %% ── STAGE 0b: TARGET WORD DUPLICATES ──────────────────────────────────────
target_dupe_mask = df_raw.duplicated("target", keep=False)
target_dupes = df_raw[target_dupe_mask]
print(f"\n=== Duplicate target words: {target_dupes['target'].nunique()} unique words in {len(target_dupes)} rows ===")
if len(target_dupes):
    print(target_dupes[["target", "duolingo_translations"]].sort_values("target").to_string())

def merge_translations(series: pd.Series) -> str:
    seen, merged = set(), []
    for cell in series.dropna():
        for tok in cell.split(","):
            tok = tok.strip()
            if tok and tok not in seen:
                seen.add(tok)
                merged.append(tok)
    return ", ".join(merged)

if len(target_dupes):
    df = df_raw.groupby("target", sort=False).agg(
        {"duolingo_translations": merge_translations}
    ).reset_index()
    print(f"\nAfter dedup: {len(df)} rows (removed {len(df_raw) - len(df)})")
else:
    df = df_raw.copy()

In [ ]:
# %% ── STAGE 0c: INTRA-ROW TRANSLATION DUPLICATES ────────────────────────────
def find_intra_dupes(cell: str) -> list[str]:
    seen, dupes = set(), []
    for t in cell.split(","):
        t = t.strip()
        if not t:
            continue
        if t in seen:
            dupes.append(t)
        else:
            seen.add(t)
    return dupes

df["_intra_dupes"] = df["duolingo_translations"].apply(find_intra_dupes)
rows_with_intra = df["_intra_dupes"].apply(bool).sum()
total_intra     = df["_intra_dupes"].apply(len).sum()

print(f"\n=== Intra-row translation duplicates: {total_intra} tokens across {rows_with_intra} rows ===")
if rows_with_intra:
    print(df[df["_intra_dupes"].apply(bool)][["target", "duolingo_translations", "_intra_dupes"]].to_string())

# Deduplicate
def dedup_trans(cell: str) -> str:
    seen, out = set(), []
    for t in cell.split(","):
        t = t.strip()
        if t and t not in seen:
            seen.add(t)
            out.append(t)
    return ", ".join(out)

df["duolingo_translations"] = df["duolingo_translations"].apply(dedup_trans)
df.drop(columns=["_intra_dupes"], inplace=True)

In [ ]:
# %% ── STAGE 0d: CROSS-ROW TRANSLATION SHARING (report only, no dedup) ───────
def parse_tlist(cell: str) -> list[str]:
    return [t.strip() for t in cell.split(",") if t.strip()] if cell else []

df["_tlist"] = df["duolingo_translations"].apply(parse_tlist)
all_tokens   = [t for tl in df["_tlist"] for t in tl]
token_counts = Counter(all_tokens)
shared = {t: c for t, c in token_counts.items() if c > 1}

print(f"\n=== Translations appearing in >1 row: {len(shared)} ===")
if shared:
    shared_df = (
        pd.DataFrame.from_dict(shared, orient="index", columns=["row_count"])
          .sort_values("row_count", ascending=False)
    )
    print(shared_df.to_string())

df.drop(columns=["_tlist"], inplace=True)
print(f"\nFinal df: {len(df)} rows")

In [ ]:
# For target stripping stats at the end
merged_groups = df_raw[df_raw.duplicated("target", keep=False)][["target", "target_raw", "duolingo_translations_raw"]]
affected_final_targets = merged_groups["target"].unique().tolist()

### Stage 1 - Morphological analysis of english target words

In [ ]:
# %% ── STAGE 1a CONFIG ────────────────────────────────────────────────────────
# Universal Dependencies pos tags

# ADJ: adjective
# ADP: adposition
# ADV: adverb
# AUX: auxiliary
# CCONJ: coordinating conjunction
# DET: determiner
# INTJ: interjection
# NOUN: noun
# NUM: numeral
# PART: particle
# PRON: pronoun
# PROPN: proper noun
# PUNCT: punctuation
# SCONJ: subordinating conjunction
# SYM: symbol
# VERB: verb
# X: other

# UD_POS_TAGS = ['ADJ','ADP','ADV','AUX','CCONJ','DET','INTJ','NOUN','NUM','PART','PRON','PROPN','PUNCT','SCONJ','SYM','VERB','X']
# UD_POS_TAGS = ['ADJ','ADV','AUX','NOUN','PROPN','VERB']

PENN_TAGS = {
    "NOUN": {"NN", "NNS", "NNP", "NNPS"},
    "VERB": {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"},
    "ADJ":  {"JJ", "JJR", "JJS"},
    "ADV":  {"RB", "RBR", "RBS"},
}
CORE_POS = {"NOUN", "VERB", "ADJ"}
ALL_POS  = {"NOUN", "VERB", "ADJ", "ADV"}
NOUN_SG  = {"NN", "NNP"}
NOUN_PL  = {"NNS", "NNPS"}

ANALYSIS_BLACKLIST: list[dict] = [
    # {"lemma": "welcome", "pos": "NOUN", "tag": "NNS"},
]

def _is_blacklisted(entry: dict) -> bool:
    for bl in ANALYSIS_BLACKLIST:
        if all(entry.get(k) == v for k, v in bl.items()):
            return True
    return False

def check_sg_pl_overlap(lemma: str, inflections: dict[str, tuple]) -> bool:
    """Returns True if any singular form == any plural form (possible lemminflect bug)."""
    sg_forms = {f for tag in NOUN_SG if tag in inflections for f in inflections[tag]}
    pl_forms = {f for tag in NOUN_PL if tag in inflections for f in inflections[tag]}
    return bool(sg_forms & pl_forms)

def getAllInflections_better(lemma: str, upos: str) -> dict[str, str]:
    allInflections = {}
    for tag in PENN_TAGS[upos]:
        allInflections[tag] = getInflection(lemma, tag=tag, inflect_oov=False)
    return allInflections

def analyze_english_word(word: str) -> list[dict]:
    """Returns list of {'pos', 'tag', 'lemma'} for all NOUN/VERB/ADJ/ADV readings."""
    word = word.strip()
    if " " in word:
        return []
    
    results = []
    lemma_dict = getAllLemmas(word)

    for upos, lemmas in lemma_dict.items():
        if upos in ALL_POS:
            for lemma in lemmas:
                inflections = getAllInflections_better(lemma, upos=upos)
                if not inflections:
                    continue
                if upos == "NOUN" and check_sg_pl_overlap(lemma, inflections):
                    sg_forms = {f for tag in NOUN_SG if tag in inflections for f in inflections[tag]}
                    pl_forms = {f for tag in NOUN_PL if tag in inflections for f in inflections[tag]}
                    logger.warning(
                        f"lemminflect SG/PL overlap: '{lemma}': sg:'{sg_forms}' pl:'{pl_forms}'"
                    )
                for tag, forms in inflections.items():
                    if word in forms:
                        entry = {"pos": upos, "penn_tag": tag, "eng_lemma": lemma}
                        if not _is_blacklisted(entry):
                            results.append(entry)
    
    # Deduplicate (same pos+tag+lemma)
    seen, deduped = set(), []
    for e in results:
        key = (e["pos"], e["penn_tag"], e["eng_lemma"])
        if key not in seen:
            seen.add(key)
            deduped.append(e)
    return deduped


In [ ]:
# %% ── STAGE 1a: WORD TYPE & LABELS ──────────────────────────────────────────
_WHITELIST_CHARS = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ- ")

def has_punctuation(word: str) -> bool:
    """True if word contains any character outside [a-zA-Z, hyphen, space]."""
    return any(c not in _WHITELIST_CHARS for c in word)

def classify_word_type(word: str, pos_set: frozenset) -> str:
    """
    Priority: punctuation > phrase > core > function > core/function
    core_pos    = {NOUN, VERB, ADJ}  (ADV intentionally excluded from core)
    non_core    = everything else including ADV
    """
    if has_punctuation(word):
        return "punctuation"
    if " " in word:
        return "phrase"
    core_in    = bool(pos_set & CORE_POS)
    ncore_in   = bool(pos_set - CORE_POS)   # ADV and anything else
    if core_in and not ncore_in:
        return "core"
    if not core_in and ncore_in:
        return "function"
    if core_in and ncore_in:
        return "core/function"
    return "OTHER" # empty pos_set — no analyses

def classify_lexeme_form(target: str, analyses: list[dict]) -> str | None:
    """
    lemma          — exactly 1 unique lemma string, identical to target word
    inflected      — exactly 1 unique lemma string, different from target word
    ambiguous      — >1 unique lemma strings, ≥1 identical to target word
    multi_inflected— >1 unique lemma strings, none identical to target word
    OTHER           — 0 analyses
    """
    if not analyses:
        return "OTHER"
    unique_lemmas = list(dict.fromkeys(a["eng_lemma"] for a in analyses))  # ordered, deduped
    n = len(unique_lemmas)
    has_match = any(l == target for l in unique_lemmas)
    if n == 1:
        return "lemma" if has_match else "inflected"
    else:
        return "ambiguous" if has_match else "multi_inflected"

def sort_analyses_matching_first(target: str, analyses: list[dict]) -> list[dict]:
    """Place entries whose lemma == target word first; preserve relative order otherwise."""
    matching    = [a for a in analyses if a["eng_lemma"] == target]
    nonmatching = [a for a in analyses if a["eng_lemma"] != target]
    return matching + nonmatching

def build_lemma_pos_dict(target: str, analyses: list[dict]) -> dict[str, set[str]]:
    """
    Build {lemma_str: set_of_POS} preserving order (matching lemmas first).
    e.g. saw → {"saw": {"NOUN","VERB"}, "see": {"VERB"}}
    """
    # Sort so matching lemmas come first
    sorted_analyses = sort_analyses_matching_first(target, analyses)
    lemma_pos: dict[str, set] = {}
    for a in sorted_analyses:
        lemma_pos.setdefault(a["eng_lemma"], set()).add(a["pos"])
    return lemma_pos  # insertion-ordered in Python 3.7+

In [ ]:
# Words that don't inject well into Polish DOM text — ambiguous, function-like,
# or produce poor matches regardless of morphological analysis.
# yes/no, ago, alone, stuff, do (done,did,does, doing), make (made, makes, making), this/that/those/there/here/these, my/yours/his/hers/theirs/ours, many/much, then/now
_BLACKLIST_WORDS: set[str] = {
    ## pronouns / determiners
    "this", "that", "these", "those",
    "I", "you", "he", "him", "she", "we", "it", "they", "them", "us", 
    "my", "mine", "your", "yours", "his", "her", "hers",
    "their", "theirs", "our", "ours", "its",
    ## quantifiers
    "many", "much",
    # "some", "any", "few", "several", "enough",
    ## discourse / response particles
    "yes", "no",
    "well", "okay", "ok",
    ## temporal/locative adverbs (highly context-dependent)
    "ago", "then", "now", 
    # "soon", "already", "still", "yet",
    # "always", "never", "often", "sometimes", "usually",
    "here", "there",
    # "everywhere", "somewhere", "nowhere",
    ## light verbs (semantically bleached, over-generate)
    "do", "make",
    # "get", "have", "take", "give", "go", "come",
    # "put", "set", "let", "keep", "run", "turn", "bring",
    ## existential / copular
    "be",
    ## misc function-like content words
    "alone", "stuff", "thing", "things",
    # "way", "ways", "time", "times", "place", "places",
    "or", "and",
    "so", "as",
    "why", "where", "what", "how", "when", "which", "whom", "whose",
    'yourselves', 'themselves', 'ourselves', 'itself', 'herself', 'himself', 'myself', 'yourself',
    'am', 'is', 'are', 'was','were', 
    'have', 'has', 'had', 
    'get', 'gets', 'got', 'gotten',
}

df['is_blacklisted'] = df['target'].apply(lambda word: bool(word in _BLACKLIST_WORDS))

In [ ]:
# %% ── STAGE 1a: RUN ──────────────────────────────────────────────────────────
df["eng_analyses"] = df["target"].apply(
    lambda w: [] if " " in w.strip() else analyze_english_word(w.strip())
)
df["eng_pos_set"] = df["eng_analyses"].apply(
    lambda a: frozenset(x["pos"] for x in a)
)
df["eng_word_type"] = df.apply(
    lambda row: classify_word_type(row["target"], row["eng_pos_set"]), axis=1
)
df["eng_lexeme_form"] = df.apply(
    lambda row: classify_lexeme_form(row["target"], row["eng_analyses"]), axis=1
)
# Sort analyses: matching lemmas first
df["eng_analyses"] = df.apply(
    lambda row: sort_analyses_matching_first(row["target"], row["eng_analyses"]), axis=1
)
# Build lemma→POS dict per row (main structure for Stage 2)
df["eng_lemma_pos"] = df.apply(
    lambda row: build_lemma_pos_dict(row["target"], row["eng_analyses"]), axis=1
)
# Capitalisation flags
df["eng_is_capitalised"] = df["target"].apply(lambda w: bool(w) and w[0].isupper())
df["eng_is_allcaps"]     = df["target"].apply(
    lambda w: bool(w) and w.replace("-","").replace(" ","").isupper() and len(w.replace("-","").replace(" ","")) > 1
)

print("Stage 1a complete.")
print(df[["target","eng_word_type","eng_lexeme_form","eng_lemma_pos","eng_is_capitalised","eng_is_allcaps"]].head(15).to_string())


# %% ── STAGE 1b: COVERAGE ─────────────────────────────────────────────────────
# skip for now


# %% ── STAGE 1c: STATS ───────────────────────────────────────────────────────
print("=" * 60)
print("STAGE 1 — STATS")
print("=" * 60)

print("\n=== eng_word_type distribution ===")
print(df["eng_word_type"].value_counts().to_string())

print("\n=== eng_lexeme_form distribution ===")
print(df["eng_lexeme_form"].value_counts(dropna=False).to_string())

print("\n=== eng_is_capitalised / eng_is_allcaps ===")
print(f"Capitalised : {df['eng_is_capitalised'].sum()}")
print(f"ALL CAPS    : {df['eng_is_allcaps'].sum()}")

# Single vs multi core POS
core_mask  = df["eng_word_type"].isin(["core", "core/function"])
single_core = df[core_mask]["eng_pos_set"].apply(lambda s: len(s & CORE_POS) == 1).sum()
multi_core  = df[core_mask]["eng_pos_set"].apply(lambda s: len(s & CORE_POS) > 1).sum()
print(f"\n=== Core POS rows: {core_mask.sum()} ===")
print(f"  Single core POS : {single_core}")
print(f"  Multi  core POS : {multi_core}")

# core/function examples
cf_mask = df["eng_word_type"] == "core/function"
print(f"\n=== core/function rows: {cf_mask.sum()} — examples ===")
print(df[cf_mask][["target","eng_pos_set","eng_analyses"]].head(15).to_string())

# Cross-row lemma duplicates (string level)
lemma_to_rows: dict[str, list] = defaultdict(list)
for idx, row in df.iterrows():
    for lemma in row["eng_lemma_pos"]:
        lemma_to_rows[lemma].append(row["target"])
eng_lemma_dupes = {l: targets for l, targets in lemma_to_rows.items()
                   if len(set(targets)) > 1}
print(f"\n=== Cross-row lemma duplicates: {len(eng_lemma_dupes)} ===")
for lemma, targets in sorted(eng_lemma_dupes.items(), key=lambda x: -len(set(x[1])))[:15]:
    print(f"  '{lemma}' → {sorted(set(targets))}")

### _old Stage 1

In [ ]:
# %% ── STAGE 1a: ENGLISH MORPH ANALYSIS ──────────────────────────────────────
#
# For each target word, build:
#   eng_analyses: list of {lemma, pos, tag} — all NOUN/VERB/ADJ readings
#   eng_pos_set:  frozenset of POS present (e.g. {"VERB", "NOUN"})
#   eng_is_phrase: bool
#   eng_pos_label: "noun" | "verb" | "adj" | "phrase" | "other" | "noun+verb" etc.
#
# Penn Treebank tags per POS:
PENN_TAGS = {
    "NOUN": {"NN", "NNS", "NNP", "NNPS"},
    "VERB": {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"},
    "ADJ":  {"JJ", "JJR", "JJS"},
}
CORE_POS = {"NOUN", "VERB", "ADJ"}
NOUN_SG  = {"NN", "NNP"}
NOUN_PL  = {"NNS", "NNPS"}

ANALYSIS_BLACKLIST: list[dict] = [
    # {"lemma": "welcome", "pos": "NOUN", "tag": "NNS"},
]

def _is_blacklisted(entry: dict) -> bool:
    for bl in ANALYSIS_BLACKLIST:
        if all(entry.get(k) == v for k, v in bl.items()):
            return True
    return False

def check_sg_pl_overlap(lemma: str, inflections: dict[str, tuple]) -> bool:
    """Returns True if any singular form == any plural form (possible lemminflect bug)."""
    sg_forms = {f for tag in NOUN_SG if tag in inflections for f in inflections[tag]}
    pl_forms = {f for tag in NOUN_PL if tag in inflections for f in inflections[tag]}
    return bool(sg_forms & pl_forms)

def analyze_english_word(word: str) -> list[dict]:
    """
    Returns list of {lemma, pos, tag} for ALL POS (not just core).
    Non-core POS entries have pos = original UPOS string from lemminflect,
    tag = None (lemminflect only covers NOUN/VERB/ADJ inflections).
    """
    word = word.strip()
    if " " in word:
        return []

    results = []
    lemma_dict = getAllLemmas(word)  # returns all UPOS

    for upos, lemmas in lemma_dict.items():
        if upos in CORE_POS:
            for lemma in lemmas:
                inflections = getAllInflections(lemma, upos=upos)
                if not inflections:
                    continue
                if upos == "NOUN" and check_sg_pl_overlap(lemma, inflections):
                    sg_forms = {f for tag in NOUN_SG if tag in inflections for f in inflections[tag]}
                    pl_forms = {f for tag in NOUN_PL if tag in inflections for f in inflections[tag]}
                    logger.warning(
                        f"lemminflect SG/PL overlap: '{lemma}': sg:'{sg_forms}' pl:'{pl_forms}'"
                    )
                for tag, forms in inflections.items():
                    if word in forms:
                        entry = {"pos": upos, "tag": tag, "lemma": lemma}
                        if not _is_blacklisted(entry):
                            results.append(entry)
        else:
            # Non-core: record lemma+pos, tag=None (no inflection table)
            for lemma in lemmas:
                entry = {"lemma": lemma, "pos": upos, "tag": None}
                if not _is_blacklisted(entry):
                    results.append(entry)

    return results


def classify_pos_label(word: str, analyses: list[dict]) -> str:
    """
    "phrase"        — target contains space
    "core"          — all pos are core (NOUN/VERB/ADJ only)
    "function"      — no core pos at all
    "core+function" — mix of core and non-core
    """
    if " " in word.strip():
        return "phrase"
    if not analyses:
        return "function"  # unknown = treated as function word

    pos_set      = {a["pos"] for a in analyses}
    has_core     = bool(pos_set & CORE_POS)
    has_noncore  = bool(pos_set - CORE_POS)

    if has_core and has_noncore:
        return "core+function"
    if has_core:
        return "core"
    return "function"


# ── Run ──────────────────────────────────────────────────────────────────────
df["eng_analyses"] = df.apply(
    lambda row: [] if " " in row["target"].strip()
    else analyze_english_word(row["target"]),
    axis=1,
)
df["eng_pos_set"] = df["eng_analyses"].apply(
    lambda a: frozenset(x["pos"] for x in a) & CORE_POS
)
df["eng_pos_label"] = df.apply(
    lambda row: classify_pos_label(row["target"], row["eng_analyses"]), axis=1
)
df["eng_lemmas_str"] = df["eng_analyses"].apply(
    lambda a: sorted({x["lemma"] for x in a})
)
df["eng_multi_lemma"] = df["eng_lemmas_str"].apply(lambda l: len(l) > 1)

# Stats for new labels
print("=== POS label distribution ===")
print(df["eng_pos_label"].value_counts())
print()

mixed = df[df["eng_pos_label"] == "core+function"]
print(f"=== core+function rows: {len(mixed)} ===")
if len(mixed):
    for _, row in mixed.head(15).iterrows():
        all_pos = sorted({a["pos"] for a in row["eng_analyses"]})
        print(f"  {row['target']:20s} POS: {all_pos}")


print("=== Rows with multiple distinct lemma strings ===")
print(df[df["eng_multi_lemma"]][["target", "eng_lemmas_str"]].head(20).to_string())

# ── Cross-row lemma duplicate check (string-level, across rows) ──────────────
lemma_to_rows: dict[str, list] = defaultdict(list)
for idx, row in df.iterrows():
    for lemma_str in row["eng_lemmas_str"]:
        lemma_to_rows[lemma_str].append((idx, row["target"]))

eng_lemma_dupes = {
    l: rows for l, rows in lemma_to_rows.items()
    # different row indices AND different target words
    if len({r[1] for r in rows}) > 1
}
print(f"\n=== English lemma cross-row duplicates: {len(eng_lemma_dupes)} (head 15)===")
for lemma, rows in sorted(eng_lemma_dupes.items(), key=lambda x: -len({r[1] for r in x[1]}))[:15]:
    targets = sorted({r[1] for r in rows})
    print(f"  '{lemma}' shared by targets: {targets}")

In [ ]:
# %% ── STAGE 1b: COVERAGE CALCULATIONS ───────────────────────────────────────
#
# For each target word that is a NOUN/VERB/ADJ, compute:
#   - all possible surface forms for each lemma (via lemminflect)
#   - which of those forms are present in the table's `target` column
#   - which Penn Treebank tags are "covered" vs "missing"
#
# We do this in a temporary unfolded DataFrame for clean computation,
# then write results back as per-row dicts.

target_set = set(df["target"])

def compute_coverage(target: str, analyses: list[dict]) -> dict:
    """
    For a given target word and its analyses, generate ALL inflections of ALL
    POS-specific lemmas, then check what fraction exist in the target table.

    Returns:
        {
          "all_forms": set[str],        — all inflected forms (deduplicated)
          "covered_forms": set[str],    — forms present in target_set
          "coverage_ratio": float,      — len(covered) / len(all) or 0
          "is_lemma": bool,             — target string == at least one lemma string
        }
    """
    if not analyses:
        return {"all_forms": set(), "covered_forms": set(),
                "coverage_ratio": 0.0, "is_lemma": False}

    all_forms: set[str] = set()
    lemma_strings = {a["lemma"] for a in analyses}

    for a in analyses:
        lemma, pos = a["lemma"], a["pos"]
        if pos not in CORE_POS:
            continue
        inflections = getAllInflections(lemma, upos=pos)
        for tag_forms in inflections.values():
            all_forms.update(f.lower() for f in tag_forms)

    covered = all_forms & {t.lower() for t in target_set}
    ratio   = len(covered) / len(all_forms) if all_forms else 0.0
    is_lem  = target.lower() in {l.lower() for l in lemma_strings}

    return {
        "all_forms":      all_forms,
        "covered_forms":  covered,
        "coverage_ratio": ratio,
        "is_lemma":       is_lem,
    }

cov_results = df.apply(
    lambda row: compute_coverage(row["target"], row["eng_analyses"]), axis=1
)

df["cov_all_forms"]      = cov_results.apply(lambda r: r["all_forms"])
df["cov_covered_forms"]  = cov_results.apply(lambda r: r["covered_forms"])
df["cov_ratio"]          = cov_results.apply(lambda r: r["coverage_ratio"])
df["cov_is_lemma"]       = cov_results.apply(lambda r: r["is_lemma"])

# ── Stats ────────────────────────────────────────────────────────────────────
core_mask = df["eng_pos_set"].apply(bool)
df_core   = df[core_mask]

print("=== Coverage ratio (core POS rows only) ===")
print(df_core["cov_ratio"].describe().round(3))
print(f"\nAverage coverage ratio : {df_core['cov_ratio'].mean():.3f}")
print(f"Rows fully covered (1.0): {(df_core['cov_ratio'] == 1.0).sum()}")
print(f"Rows partially covered  : {((df_core['cov_ratio'] > 0) & (df_core['cov_ratio'] < 1)).sum()}")
print(f"Rows not covered at all : {(df_core['cov_ratio'] == 0.0).sum()}")

print(f"\n=== is_lemma flag ===")
print(f"Target IS its own lemma     : {df['cov_is_lemma'].sum()} / {len(df)}")
print(f"Target is NOT its own lemma : {(~df['cov_is_lemma']).sum()} / {len(df)}")

# ── Worst coverage — useful for deciding what to add to vocab ────────────────
print("\n=== Rows with partial coverage (missing some inflections) (head 10)===")
partial = df_core[(df_core["cov_ratio"] > 0) & (df_core["cov_ratio"] < 1)].copy()
partial["missing_forms"] = partial.apply(
    lambda row: sorted(row["cov_all_forms"] - row["cov_covered_forms"]), axis=1
)
print(partial[["target", "cov_ratio", "missing_forms"]].sort_values("cov_ratio").head(10).to_string())

In [ ]:
# %% ── STAGE 1c: ENGLISH STATS SUMMARY ───────────────────────────────────────
print("=" * 60)
print("STAGE 1 — ENGLISH MORPHOLOGICAL ANALYSIS SUMMARY")
print("=" * 60)
print(f"Total rows              : {len(df)}")
print(f"Phrase rows             : {(df['eng_pos_label'] == 'phrase').sum()}")
print(f"Other (no core POS)     : {(df['eng_pos_label'] == 'other').sum()}")
print(f"Rows with ≥1 core POS   : {df['eng_pos_set'].apply(bool).sum()}")
print()

# Core POS: single vs multiple
single_core = df["eng_pos_set"].apply(lambda s: len(s) == 1)
multi_core  = df["eng_pos_set"].apply(lambda s: len(s) > 1)
print(f"  Single core POS rows  : {single_core.sum()}")
print(f"  Multi  core POS rows  : {multi_core.sum()}")
if multi_core.sum():
    print("  Multi-POS distribution:")
    print(df[multi_core]["eng_pos_label"].value_counts().to_string())
print()

# POS breakdown
print("=== POS label breakdown ===")
print(df["eng_pos_label"].value_counts().to_string())
print()

# Per-POS lemma stats (cross-row dupes already computed above)
for pos in ("NOUN", "VERB", "ADJ"):
    pos_rows = df[df["eng_pos_set"].apply(lambda s: pos in s)]
    lemmas   = [a["lemma"] for row in pos_rows["eng_analyses"] for a in row if a["pos"] == pos]
    n_unique = len(set(lemmas))
    print(f"  {pos}: {len(pos_rows)} rows, {n_unique} unique lemmas")

print(f"\n=== Cross-row lemma duplicates: {len(eng_lemma_dupes)} lemmas shared across rows ===")

# Multi-lemma rows
print(f"\n=== Rows with multiple distinct lemma strings: {df['eng_multi_lemma'].sum()} ===")

# Coverage summary (core POS only)
if df["cov_ratio"].notna().any():
    print(f"\n=== Form coverage (core POS rows) ===")
    print(f"  Average coverage ratio : {df_core['cov_ratio'].mean():.3f}")
    print(f"  Fully covered rows     : {(df_core['cov_ratio'] == 1.0).sum()}")
    print(f"  Partial coverage rows  : {((df_core['cov_ratio'] > 0) & (df_core['cov_ratio'] < 1.0)).sum()}")
    print(f"  Zero coverage rows     : {(df_core['cov_ratio'] == 0.0).sum()}")
    print(f"\n=== is_lemma flag ===")
    print(f"  Target is own lemma    : {df['cov_is_lemma'].sum()}")
    print(f"  Target is inflected form: {(~df['cov_is_lemma'] & df['eng_pos_set'].apply(bool)).sum()}")

# Analyses count distribution
n_analyses = df["eng_analyses"].apply(len)
print(f"\n=== Lemma×tag entries per row ===")
print(n_analyses.describe().round(2))
print(f"Rows with 0 analyses (unknown to lemminflect): {(n_analyses == 0).sum()}")

### Stage 1 - supplement

eng_word_type_t — translation-based POS inference for eng_word_type OTHER

In [ ]:
import re

# Morfeusz POS → normalised POS mapping
MORF_POS_MAP: dict[str, str] = {
    # NOUN-like
    "subst":  "NOUN",
    "depr":   "NOUN",
    "num":    "NOUN",
    "numcol": "NOUN",
    # ADJ-like
    "adj":    "ADJ",
    "adja":   "ADJ",
    "adjp":   "ADJ",
    "adjc":   "ADJ",
    # VERB-like (finite/non-finite)
    "fin":    "VERB",
    "inf":    "VERB",
    "impt":   "VERB",
    "praet":  "VERB",
    # VERB-like (participial)
    "pact":   "VERB",
    "pcon":   "VERB",
    "ger":    "VERB",
    # VERB-like (passive/anterior)
    "imps":   "VERB",
    "ppas":   "VERB",
    "pant":   "VERB",
    # ADVERB - non core
    "adv":    "ADV",
}
# Discard any morfeusz record with these features or qualifiers
MORF_BANNED_FEATURES = {"nazwisko", "imię"}
MORF_BANNED_QUALIFIERS = {"pot.", "wulg.", "środ.", "arch.", "daw.", "pogard.", "przest.", "roln.", "gwar."} #"rzad.", "z_d."}

def morf_analyse_clean(word: str) -> list[dict]:
    """
    Run morfeusz on word, return list of clean records:
        {'lemma_raw': str, 'morf_tag': str, 'pos': str, 'lemma': str}
    Discards:
      - records with banned features (features field)
      - records with banned qualifiers (flags field)
      - records where norm_pos is None (POS not in MORF_POS_MAP)
      - records with tag == 'ign'
    """
    results = []
    for _, _, (orth, lemma_raw, tag, features, flags) in morf.analyse(word):
        # flags may be a single string in a list
        if len(flags)==1:
            flags = flags[0].split(',')

        if tag == 'ign':
            continue
        feat_set = set(features) if features else set()
        flag_set = set(flags)    if flags    else set()
        if feat_set & MORF_BANNED_FEATURES:
            continue
        if flag_set & MORF_BANNED_QUALIFIERS:
            continue
        morf_class = tag.split(':')[0]
        norm_pos   = MORF_POS_MAP.get(morf_class)
        if norm_pos is None:
            continue
        lemma = lemma_raw.split(':')[0]
        results.append({
            'lemma_raw':  lemma_raw,
            'morf_tag':   tag,
            'pos':   norm_pos,
            'lemma': lemma,    
        })
    return results


def analyse_translations(translations_str: str) -> list[set[str]]:
    """
    Returns list of, per token, set of possible pos tags.
    """
    results = []
    _RE_SIE_INFERRED = re.compile(r'\bsię\b')
    for token in [t.strip() for t in translations_str.split(',') if t.strip()]:
        _token = token
        token = _RE_SIE_INFERRED.sub('', token).strip()
        if not token:
            results.append(set()) # PART 'się'
        elif ' ' in token:
            results.append({'phrase'})
        else:
            results.append({rec['pos'] for rec in morf_analyse_clean(token)})
    return results


def classify_token_word_type(tag_set: set) -> str:
    if 'phrase' in tag_set:
        return 'phrase'
    in_core  = tag_set & CORE_POS
    out_core = tag_set - CORE_POS
    if in_core and not out_core:
        return "core"
    if in_core and out_core:
        return "ambiguous"
    return "function"   # only out_core tags


def summerise_translations_word_type(token_types: list[str]) -> dict:
    if all(t == "phrase" for t in token_types):
        return "phrase"

    has_core = any(t == "core" for t in token_types)
    has_non_core = any(t not in {"core", "phrase"} for t in token_types)

    if has_core and not has_non_core:
        return "core"
    if has_core and has_non_core:
        return "ambiguous"
    return "function"


In [ ]:
df["t_word_types"] = df["duolingo_translations"].apply(lambda s: [classify_token_word_type(tags) for tags in analyse_translations(s)])
df["t_word_type"] = df["t_word_types"].apply(summerise_translations_word_type)

In [ ]:
other_mask = df["eng_word_type"] == "OTHER"

# ── Stats ─────────────────────────────────────────────────────────────────────
print("=== eng_word_type_t distribution (OTHER rows only) ===")
print(df[other_mask]["t_word_type"].value_counts(dropna=False).to_string())

core_t   = (other_mask) & (df["t_word_type"] == "core")
ambig_t  = (other_mask) & (df["t_word_type"] == "ambiguous")
func_t   = (other_mask) & (df["t_word_type"] == "function")
phrase_t = (other_mask) & (df["t_word_type"] == "phrase")

print(f"\nOTHER rows flagged for Stage 2 scraping (core)   : {core_t.sum()}")
print(f"OTHER rows ambiguous (skip Stage 2)                : {ambig_t.sum()}")
print(f"OTHER rows function  (skip Stage 2)                : {func_t.sum()}")
print(f"OTHER rows phrases  (skinclude in Stage 2?)        : {phrase_t.sum()}")

print("\n=== Sample: OTHER → core_t (will be scraped) ===")
print(df[core_t][["target", "duolingo_translations", "t_word_type"]].head(15).to_string())

print("\n=== Sample: OTHER → ambiguous_t ===")
print(df[ambig_t][["target", "duolingo_translations", "t_word_type"]].head(10).to_string())

print("\n=== Sample: OTHER → function_t ===")
print(df[func_t][["target", "duolingo_translations", "t_word_type"]].head(10).to_string())

print("\n=== Sample: OTHER → phrase_t ===")
print(df[func_t][["target", "duolingo_translations", "t_word_type"]].head(10).to_string())

### Stage 2 - translations

DIKI scraping translations

In [ ]:
import json, random, time, re
import requests
from requests.exceptions import HTTPError
from bs4 import BeautifulSoup
from collections import OrderedDict
from pathlib import Path

DIKI_CHECKPOINT  = Path(INPUT_CSV).with_name("diki_checkpoint.json")
TOP_N_TOTAL      = 10
BASE_DELAY       = 0.5 #1.3
JITTER           = 0.2 #0.8
MAX_RETRIES      = 3

HEADERS_POOL = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3 Safari/605.1.15',
    'Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0',
]

# POS accepted from diki → normalised POS
DIKI_POS_MAP: dict[str, str] = {
    "rzeczownik": "NOUN",
    "liczebnik":  "NOUN",   # numerals treated as nouns per plan
    "czasownik":  "VERB",
    "przymiotnik":"ADJ",
    "przysłówek": "ADV",
}

WORDS_TO_TRANSLATE_MASK = (df["eng_word_type"].isin(["core", "core/function"]) | ((df["eng_word_type"]=='OTHER') & (df["t_word_type"].isin(["core", "ambiguous"])))) & ~df['is_blacklisted']

##### helpers

In [ ]:
def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({
        'User-Agent':      random.choice(HEADERS_POOL),
        'Accept-Language': 'pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Referer':         'https://www.diki.pl/',
    })
    try:
        s.get('https://www.diki.pl/', timeout=10)
        time.sleep(1.0)
    except Exception:
        pass
    return s


def _get_en_pl_results_container(soup: BeautifulSoup):
    """
    Find the diki-results-container that follows the en-pl section header.
    Structure:
        <div>
          <div id="en-pl"></div>
          <h2 class="dictionarySectionHeader">...</h2>
        </div>
        <div class="diki-results-container">...</div>   ← want this
    Finds first div#en-pl, walks up to its parent, returns the next sibling
    diki-results-container.
    """
    en_pl_div = soup.find('div', id='en-pl')
    if en_pl_div is None:
        return None
    parent = en_pl_div.parent
    if parent is None:
        return None
    # Walk siblings of parent to find next diki-results-container
    sibling = parent.find_next_sibling()
    while sibling is not None:
        if (hasattr(sibling, 'get') and
                'diki-results-container' in sibling.get('class', [])):
            return sibling
        sibling = sibling.find_next_sibling()
    return None


def _parse_hw_text(span) -> str:
    """
    Extract text from a span.hw element.
    The text may be inside a nested <a> or directly in the span.
    Returns the raw text, preserving capitalisation.
    """
    return span.get_text(separator=' ', strip=True)


def _get_hw_strings(hws_div) -> list[str]:
    """
    Return all hw strings from a dictionaryEntity's hws div.
    Multiple <span class="hw"> can exist (e.g. 'March', 'Mar.').
    """
    return [_parse_hw_text(sp)
            for sp in hws_div.find_all('span', class_='hw')]


def _parse_meaning_li(li_tag) -> list[str]:
    """
    Extract individual translation tokens from a <li class="meaningXXX"> element.

    Strategy:
      1. Collect all span.hw elements — each is one translation candidate.
      2. For each span text: remove any (...) or [...] commentary.
      3. Trim leading/trailing whitespace and punctuation.
      4. Skip if empty or contains inner whitespace (phrase).
      5. Allow hyphens within token (e.g. "t-shirt").
    """
    _RE_BRACKETS     = re.compile(r'\([^)]*\)|\[[^\]]*\]')   # () and [] content
    _RE_SIE          = re.compile(r'\bsię\b')               # let czasowniki zwrotne prevail
    _RE_LEAD_TRAIL   = re.compile(r'^[\s\W]+|[\s\W]+$')       # leading/trailing non-word chars
    _RE_INNER_SPACE  = re.compile(r'\s')                       # any whitespace within token
    
    tokens: list[str] = []
    for hw_span in li_tag.find_all('span', class_='hw', recursive=True):
        raw = _parse_hw_text(hw_span)
        # Remove parenthetical/bracket commentary
        cleaned = _RE_BRACKETS.sub('', raw)
        # treat czasowniki zwrotne as if they weren't
        cleaned = _RE_SIE.sub('', cleaned)
        # Trim lead/trail punctuation and whitespace
        cleaned = _RE_LEAD_TRAIL.sub('', cleaned)
        if not cleaned:
            continue
        # Skip phrases (inner whitespace)
        if _RE_INNER_SPACE.search(cleaned):
            continue
        tokens.append(cleaned)
    return tokens


def scrape_lemma(
    lemma: str,
    allowed_pos: set[str],   # normalised POS {"NOUN","VERB",...}; empty = accept all
    session: requests.Session,
) -> list[dict]:
    """
    Scrape diki EN→PL for `lemma`.
    Returns list of {'pos': str, 'translation': str, 'column': 'primary'|'secondary'}.

    Filtering rules (applied in order):
      - diki POS not in DIKI_POS_MAP              → skip entire meanings block
      - mapped POS not in allowed_pos (if non-empty) → skip entire meanings block
      - translation is phrase (inner space)        → skip (handled in _parse_meaning_li)
      - per-POS counter reaches TOP_N_TOTAL        → stop accepting that POS
    """
    word_query = lemma.strip().replace(' ', '+')
    soup  = None
    delay = BASE_DELAY

    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(
                f'https://www.diki.pl/slownik-angielskiego?q={word_query}',
                timeout=12,
            )
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'html.parser')
            break

        except HTTPError as e:
            status = e.response.status_code

            if status == 404:
                return 404

            if status in {403, 429}:
                retry_after = e.response.headers.get("Retry-After")
                if retry_after:
                    wait_time = int(retry_after)
                    print(f"  [retry {attempt+1}/{MAX_RETRIES}] {lemma}: Blocked ({status}). Waiting {wait_time}s.")
                    time.sleep(wait_time)
                else:
                    print(f"  [retry {attempt+1}/{MAX_RETRIES}] {lemma}: Blocked ({status}). Waiting 15 min.")
                    time.sleep(15 * 60)
            else:
                raise

        except Exception as e:
            print(f"  [retry {attempt+1}/{MAX_RETRIES}] {lemma}: {e}")
            time.sleep(3)
            if attempt < MAX_RETRIES - 1:
                time.sleep(delay + random.uniform(-JITTER, JITTER))
                delay *= 3
                session.headers.update({'User-Agent': random.choice(HEADERS_POOL)})
            else:
                return None

    if soup is None:
        return None

    # ── Locate the EN→PL results container ───────────────────────────────────
    container = _get_en_pl_results_container(soup)
    if container is None:
        container = soup   # fallback: search whole page

    lemma_norm = lemma.strip()   # case-sensitive match

    results: list[dict]        = []
    pos_counters: dict[str, int] = {}   # norm_pos → n translations collected

    # ── Iterate dictionaryEntity divs ─────────────────────────────────────────
    for entity in container.find_all('div', class_='dictionaryEntity'):

        # 1. Check if this entity is for our lemma (case-sensitive)
        hws_div = entity.find('div', class_='hws')
        if hws_div is None:
            continue
        hw_strings = _get_hw_strings(hws_div)
        if lemma_norm not in hw_strings:
            continue

        # 2. Walk direct children to find POS headers and meaning lists
        #    partOfSpeechSectionHeader and foreignToNativeMeanings are siblings
        #    at the same level inside dictionaryEntity.
        current_norm_pos: str | None = None

        for child in entity.children:
            if not hasattr(child, 'get'):
                continue   # skip NavigableString (whitespace etc.)

            child_classes = child.get('class', [])

            # ── POS section header ────────────────────────────────────────────
            if 'partOfSpeechSectionHeader' in child_classes:
                pos_span = child.find('span', class_='partOfSpeech')
                if pos_span is None:
                    current_norm_pos = None
                    continue
                pos_raw          = pos_span.get_text(strip=True).lower()
                mapped           = DIKI_POS_MAP.get(pos_raw)       # None if not in map
                current_norm_pos = mapped   # may be None → will skip meanings

                if mapped and allowed_pos and mapped not in allowed_pos:
                    current_norm_pos = None  # POS not allowed for this lemma lookup
                continue

            # ── Meanings list ─────────────────────────────────────────────────
            if 'foreignToNativeMeanings' in ' '.join(child_classes):
                if current_norm_pos is None:
                    continue   # POS was skipped or unknown

                # Check per-POS cap
                if pos_counters.get(current_norm_pos, 0) >= TOP_N_TOTAL:
                    continue

                for li in child.find_all('li', class_=re.compile(r'^meaning\d+')):
                    if pos_counters.get(current_norm_pos, 0) >= TOP_N_TOTAL:
                        break   # cap reached mid-list

                    tokens = _parse_meaning_li(li)
                    for token in tokens:
                        if pos_counters.get(current_norm_pos, 0) >= TOP_N_TOTAL:
                            break
                        # Deduplicate within this lemma scrape
                        already = [r['translation'] for r in results
                                   if r['pos'] == current_norm_pos]
                        if token in already:
                            continue
                        n = pos_counters.get(current_norm_pos, 0)
                        results.append({
                            'pos':         current_norm_pos,
                            'translation': token,
                        })
                        pos_counters[current_norm_pos] = n + 1
                continue

    return results

##### evaluate cache

In [ ]:
diki_cache: dict[str, list[dict]] = {}
if DIKI_CHECKPOINT.exists():
    raw = json.loads(DIKI_CHECKPOINT.read_text(encoding='utf-8'))
    diki_cache = raw
    print(f"Loaded checkpoint: {len(diki_cache)} lemmas cached.")

# Build global lemma→allowed_pos map with fallback for empty eng_lemma_pos
global_lemma_pos: dict[str, set] = {}
for _, row in df[WORDS_TO_TRANSLATE_MASK].iterrows():
    lemma_pos: dict = row["eng_lemma_pos"]
    if lemma_pos:
        for lemma, pos_set in lemma_pos.items():
            global_lemma_pos.setdefault(lemma, set()).update(pos_set)
    else:
        # Fallback: scrape the target word directly, accept all POS
        target = row["target"].strip()
        if target not in global_lemma_pos:
            global_lemma_pos[target] = set()

to_scrape = [l for l in global_lemma_pos if l not in diki_cache]
print(f"Lemmas to scrape: {len(to_scrape)}  (cached: {len(diki_cache)})")

In [ ]:
n_requests = len(to_scrape)
# Time estimate
SESSION_ROTATE = 17   # extra ~2s per rotation
n_rotations    = n_requests // SESSION_ROTATE

total_seconds_min = n_requests * (BASE_DELAY - JITTER)
total_seconds_max = n_requests * (BASE_DELAY + JITTER) + n_rotations * 2.0
total_seconds_avg = n_requests * BASE_DELAY + n_rotations * 2.0

def fmt_time(s: float) -> str:
    m, sec = divmod(int(s), 60)
    return f"{m}m {sec}s"

print("=" * 50)
print("DRY RUN — STAGE 2 SCRAPING ESTIMATE")
print("=" * 50)
print(f"Core rows to process         : {WORDS_TO_TRANSLATE_MASK.sum()}")
print(f"Unique English lemmas        : {n_requests}")
print(f"Session rotations            : {n_rotations}")
print()
print(f"Estimated time (min)         : {fmt_time(total_seconds_min)}")
print(f"Estimated time (avg)         : {fmt_time(total_seconds_avg)}")
print(f"Estimated time (max)         : {fmt_time(total_seconds_max)}")
print()

### run scraper

In [ ]:
# If word has not matching translations, return []
# If word returns 404 - doesn't exist in dictionary, skip

from tqdm import tqdm

session = make_session()
session_requests = 0
fail_streak = 0

pbar = tqdm(to_scrape, desc="Scraping", unit="lemma")
for lemma in pbar:
    session_requests += 1
    if session_requests % 20 == 0:
        session = make_session()

    allowed = global_lemma_pos[lemma]
    results = scrape_lemma(lemma, allowed, session)

    if isinstance(results, dict) and results.get('status_code') == 404:
         tqdm.write(f"Translation for:'{lemma}' not found.")
         continue
    elif results is None:
        fail_streak += 1
        tqdm.write(f"FAILED (network/block): {lemma}")
        if fail_streak >= 3:
            # raise RuntimeError("Too many consecutive request failures (likely blocked)")
            tqdm.write(f"Too many consecutive request failures (likely blocked). Waiting 20 min...")
            time.sleep(20*60)
        continue
    else:
        fail_streak = 0
    
    diki_cache[lemma] = results

    DIKI_CHECKPOINT.write_text(
        json.dumps(diki_cache, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    pbar.set_postfix(lemma=lemma, found=len(results))
    time.sleep(BASE_DELAY + random.uniform(-JITTER, JITTER))

print(f"\nScraping complete. In cache:")
total = len(diki_cache)
empty = sum(1 for v in diki_cache.values() if v == [])
non_empty = sum(1 for v in diki_cache.values() if v)
print(f"total: {total}")
print(f"empty: {empty}")
print(f"non-empty: {non_empty}")

### _old Stage 2 translations - (wictionary + diki supplement)

In [ ]:
# %% ── STAGE 1d: WIKTIONARY TRANSLATIONS STATS (N/V/A ONLY) ──────────────────
# Run after adding wiktionary_translations column to df.
# Assumes df has: target, eng_pos_set, wiktionary_translations

assert "wiktionary_translations" in df.columns and "wiktionary_translations2" in df.columns, "Run wiktionary_translate.py first and reload CSV"

CORE_POS_MASK = df["eng_pos_set"].apply(bool)
df_core = df[CORE_POS_MASK].copy()

print(f"=== Wiktionary stats — core POS rows only ({len(df_core)} / {len(df)}) ===\n")

# Coverage: how many rows have any wiktionary translation
has_wikt = df_core["wiktionary_translations"].fillna("").str.strip().astype(bool)
print(f"Rows WITH wiktionary translation    : {has_wikt.sum()} ({has_wikt.mean()*100:.1f}%)")
print(f"Rows WITHOUT wiktionary translation : {(~has_wikt).sum()}")

# Translation count per row
def count_translations(cell: str) -> int:
    return len([t for t in cell.split(",") if t.strip()]) if cell else 0

df_core["_wikt_count"] = df_core["wiktionary_translations"].fillna("").apply(count_translations)
df_core["_wikt2_count"] = df_core["wiktionary_translations2"].fillna("").apply(count_translations) \
    if "wiktionary_translations2" in df_core.columns else 0

print(f"\n=== Translations per row (primary column) ===")
print(df_core["_wikt_count"].describe().round(2))
print(f"Rows with 0     : {(df_core['_wikt_count'] == 0).sum()}")
print(f"Rows with 1-3   : {((df_core['_wikt_count'] >= 1) & (df_core['_wikt_count'] <= 3)).sum()}")
print(f"Rows with 4+    : {(df_core['_wikt_count'] >= 4).sum()}")

if "wiktionary_translations2" in df_core.columns:
    has_overflow = df_core["_wikt2_count"] > 0
    print(f"\nRows with overflow translations (col2): {has_overflow.sum()}")
    print(df_core["_wikt2_count"].describe().round(2))

# Breakdown by English POS
print("\n=== Coverage by English POS ===")
for pos in ("NOUN", "VERB", "ADJ"):
    mask = df_core["eng_pos_set"].apply(lambda s: pos in s)
    subset = df_core[mask]
    covered = subset["_wikt_count"].gt(0).sum()
    print(f"  {pos}: {covered}/{len(subset)} rows have wiktionary translations "
          f"({covered/len(subset)*100:.1f}%)" if len(subset) else f"  {pos}: 0 rows")

# Not-found words (core POS only)
not_found_core = df_core[df_core["_wikt_count"] == 0]["target"].tolist()
print(f"\n=== Core POS words missing from wiktionary: {len(not_found_core)} ===")
print(not_found_core[:40])

# ── Missing words breakdown: lemma vs inflected vs other ─────────────────────
print("\n=== Missing wiktionary translations — breakdown ===")

missing_mask = df["wiktionary_translations"].fillna("").str.strip() == ""
df_missing = df[missing_mask].copy()
print(f"Total missing: {len(df_missing)}")

# Classify each missing word
def classify_missing(row) -> str:
    if row["eng_pos_label"] == "phrase":
        return "phrase"
    if row["eng_pos_label"] == "other" or not row["eng_pos_set"]:
        return "non_core_pos"
    if row["cov_is_lemma"]:
        return "lemma"
    return "inflected_form"

df_missing["_miss_class"] = df_missing.apply(classify_missing, axis=1)

counts = df_missing["_miss_class"].value_counts()
print(counts.to_string())
print()

# Per-category word lists
for category in ["lemma", "inflected_form", "phrase", "non_core_pos"]:
    subset = df_missing[df_missing["_miss_class"] == category]["target"].tolist()
    if subset:
        print(f"--- {category} ({len(subset)}) ---")
        print(", ".join(sorted(subset)))
        print()


# Flag lemma words missing wiktionary translations — candidates for diki
df["needs_diki"] = (
    missing_mask &                                  # no wiktionary translation
    df["cov_is_lemma"] &                            # is a base/lemma form
    df["eng_pos_set"].apply(bool)                   # has at least one core POS
)
print(f"\n=== Words flagged for diki scraping: {df['needs_diki'].sum()} ===")
print(df[df["needs_diki"]]["target"].tolist())

In [ ]:
# %% ── STAGE 1e: DIKI SCRAPING FOR FLAGGED WORDS ─────────────────────────────
# Runs diki_translate logic inline — only processes rows where needs_diki=True.
# Saves checkpoint so interrupted runs resume automatically.

import json, random, time, re
import requests
from bs4 import BeautifulSoup
from collections import OrderedDict

# ── Config ───────────────────────────────────────────────────────────────────
DIKI_CHECKPOINT = Path(INPUT_CSV).with_name("diki_checkpoint.json")
TOP_N_PER_POS   = 3
BASE_DELAY      = 1.3
JITTER          = 0.8
MAX_RETRIES     = 4

HEADERS_POOL = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3 Safari/605.1.15',
    'Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0',
]

POS_MAP_DIKI = {
    'rzeczownik':   'n',
    'czasownik':    'v',
    'przymiotnik':  'a',
}


# ── Session setup ─────────────────────────────────────────────────────────────
def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({
        'User-Agent':      random.choice(HEADERS_POOL),
        'Accept-Language': 'pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Referer':         'https://www.diki.pl/',
    })
    try:
        s.get('https://www.diki.pl/', timeout=10)
        time.sleep(1.0)
    except Exception:
        pass
    return s

_RE_ANY_PAREN = re.compile(r'\s*\([^)]*\)')   # all parentheticals anywhere in string

def clean_diki_translation(text: str) -> str:
    """
    Remove ALL parenthetical comments from a translation token.
    e.g. "powód (osoba, wydarzenie, rzecz powodująca coś)" → "powód"
         "napój gazowany (np. cola)" → "napój gazowany"
         "wziąć" → "wziąć"  (no change)
    """
    return _RE_ANY_PAREN.sub('', text).strip().strip(',').strip()

def _scrape_diki(word: str, session: requests.Session) -> tuple[str, str]:
    """
    Returns (primary, overflow) translation strings.
    primary  = top TOP_N_PER_POS translations per POS, formatted "(pos) word"
    overflow = remaining translations beyond TOP_N_PER_POS per POS
    """
    word_norm  = word.strip().lower().replace('\u00a0', ' ')
    word_query = word.strip().replace(' ', '+')
    soup  = None
    delay = BASE_DELAY

    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(
                f'https://www.diki.pl/slownik-angielskiego?q={word_query}',
                timeout=12,
            )
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'html.parser')
            break
        except Exception as e:
            print(f"  [retry {attempt+1}/{MAX_RETRIES}] {word}: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(delay + random.uniform(-JITTER, JITTER))
                delay *= 2
                session.headers.update({'User-Agent': random.choice(HEADERS_POOL)})
            else:
                return '', ''

    if soup is None:
        return '', ''

    by_pos: dict[str, list[str]] = OrderedDict()

    for entity in soup.find_all('div', class_=re.compile(r'\bdictionaryEntity\b')):
        # Verify this entity is for our word
        hws_block = entity.find(class_=re.compile(r'\bhws\b'))
        if not hws_block:
            continue
        hw_words = [hw.get_text().strip().lower()
                    for hw in hws_block.find_all(class_=re.compile(r'\bhw\b'))]
        if word_norm not in hw_words:
            continue

        # ── Walk children to handle multiple POS sections in one entity ──────
        # Structure can be:
        #   [partOfSpeech]  → [foreignToNativeMeanings]
        #   [partOfSpeechSectionHeader] → [foreignToNativeMeanings]
        #   [partOfSpeechSectionHeader] → [foreignToNativeMeanings]  (repeats)
        #
        # Strategy: walk all direct+nested children linearly, track current_pos,
        # assign meanings to whatever current_pos is active.

        current_pos = ''  # default: no POS label

        # First check for top-level partOfSpeech (covers simple single-POS entities)
        top_pos = entity.find(class_=re.compile(r'\bpartOfSpeech\b'))
        if top_pos:
            pos_raw     = top_pos.get_text().strip().lower()
            current_pos = POS_MAP_DIKI.get(pos_raw, pos_raw) if pos_raw else ''

        # Now find ALL meaning sections, each potentially preceded by a section header
        # We look for partOfSpeechSectionHeader elements and foreignToNativeMeanings
        # in document order within this entity.
        for element in entity.descendants:
            # Skip non-tag nodes
            if not hasattr(element, 'get'):
                continue

            cls = ' '.join(element.get('class', []))

            # POS section header — updates current_pos for subsequent meanings
            if re.search(r'\bpartOfSpeechSectionHeader\b', cls):
                pos_raw     = element.get_text().strip().lower()
                current_pos = POS_MAP_DIKI.get(pos_raw, pos_raw) if pos_raw else current_pos
                continue

            # Meanings list — collect under current_pos
            if re.search(r'\bforeignToNativeMeanings\b', cls):
                by_pos.setdefault(current_pos, [])
                for m in element.find_all('li', class_=re.compile(r'^meaning\d+')):
                    for span in m.find_all('span', class_=re.compile(r'\bhw\b')):
                        part = clean_diki_translation(span.get_text().strip())
                        if part and part not in by_pos[current_pos]:
                            by_pos[current_pos].append(part)
                continue

    # Split into primary / overflow columns
    primary, overflow = [], []
    for pos, trans_list in by_pos.items():
        prefix = f'({pos}) ' if pos else ''
        for t in trans_list[:TOP_N_PER_POS]:
            primary.append(f'{prefix}{t}')
        for t in trans_list[TOP_N_PER_POS:]:
            overflow.append(f'{prefix}{t}')

    return ', '.join(primary), ', '.join(overflow)

# ── Load checkpoint ───────────────────────────────────────────────────────────
diki_cache: dict[str, tuple[str, str]] = {}
if DIKI_CHECKPOINT.exists():
    raw = json.loads(DIKI_CHECKPOINT.read_text(encoding='utf-8'))
    diki_cache = {w: tuple(v) for w, v in raw.items()}
    print(f"Loaded checkpoint: {len(diki_cache)} words already cached.")

# ── Scrape flagged words ──────────────────────────────────────────────────────
flagged_idx = df[df["needs_diki"]].index.tolist()
print(f"Words to scrape: {len(flagged_idx)}  "
      f"(already cached: {sum(1 for i in flagged_idx if df.at[i,'target'].lower() in diki_cache)})")

session = make_session()
session_requests = 0

for count, idx in enumerate(flagged_idx, 1):
    word      = df.at[idx, "target"]
    word_key  = word.lower()

    if word_key in diki_cache:
        print(f"[{count:3d}/{len(flagged_idx)}] (cached) {word}")
        continue

    session_requests += 1
    if session_requests % 50 == 0:
        session = make_session()
        print(f"  [session rotated at {session_requests} requests]")

    primary, overflow = _scrape_diki(word, session)
    diki_cache[word_key] = (primary, overflow)

    # Save checkpoint after every word
    DIKI_CHECKPOINT.write_text(
        json.dumps(diki_cache, ensure_ascii=False, indent=2), encoding='utf-8'
    )

    status = "✓" if primary else "✗ NOT FOUND"
    print(f"[{count:3d}/{len(flagged_idx)}] {status} {word:30s} → {primary[:80]}")
    time.sleep(BASE_DELAY + random.uniform(-JITTER, JITTER))

# ── Write results back to df ──────────────────────────────────────────────────
df["diki_translations"]  = ""
df["diki_translations2"] = ""

for idx in flagged_idx:
    word_key = df.at[idx, "target"].lower()
    primary, overflow = diki_cache.get(word_key, ('', ''))
    df.at[idx, "diki_translations"]  = primary
    df.at[idx, "diki_translations2"] = overflow

# Summary
found    = df[df["needs_diki"]]["diki_translations"].str.strip().astype(bool).sum()
total    = df["needs_diki"].sum()
still_missing = df[df["needs_diki"] & (df["diki_translations"].str.strip() == "")]["target"].tolist()
print(f"\n=== Diki results: {found}/{total} words found ===")
if still_missing:
    print(f"Still missing ({len(still_missing)}): {still_missing}")

### Stage 3

#### add translations, overwrite calendar translations, analyse learned inflections

In [ ]:
def build_row_diki(row: pd.Series) -> dict:
    """
    Build translations for each target word lemma from dictionary cache
    Returns {lemma: [{'pos','translation','column'}, ...]} for this row.
    Falls back to target word if eng_lemma_pos is empty.
    """
    eng_lemma_pos: dict = row['eng_lemma_pos']

    cache_keys = eng_lemma_pos.keys() or {row["target"].strip()}
    valid_cache_keys = diki_cache.keys() & cache_keys
    return {k: diki_cache[k] for k in valid_cache_keys if diki_cache[k]}


df["diki_translations_raw"] = df.apply(build_row_diki, axis=1)

In [ ]:
CALENDAR_PL: dict[str, list[dict]] = {
    # days
    "Monday":    [{"eng_lemma": "Monday",    "pos": "NOUN", "translation": "poniedziałek"}],
    "Tuesday":   [{"eng_lemma": "Tuesday",   "pos": "NOUN", "translation": "wtorek"}],
    "Wednesday": [{"eng_lemma": "Wednesday", "pos": "NOUN", "translation": "środa"}],
    "Thursday":  [{"eng_lemma": "Thursday",  "pos": "NOUN", "translation": "czwartek"}],
    "Friday":    [{"eng_lemma": "Friday",    "pos": "NOUN", "translation": "piątek"}],
    "Saturday":  [{"eng_lemma": "Saturday",  "pos": "NOUN", "translation": "sobota"}],
    "Sunday":    [{"eng_lemma": "Sunday",    "pos": "NOUN", "translation": "niedziela"}],
    # months
    "January":   [{"eng_lemma": "January",   "pos": "NOUN", "translation": "styczeń"}],
    "February":  [{"eng_lemma": "February",  "pos": "NOUN", "translation": "luty"}],
    "March":     [{"eng_lemma": "March",     "pos": "NOUN", "translation": "marzec"}],
    "April":     [{"eng_lemma": "April",     "pos": "NOUN", "translation": "kwiecień"}],
    "May":       [{"eng_lemma": "May",       "pos": "NOUN", "translation": "maj"}],
    "June":      [{"eng_lemma": "June",      "pos": "NOUN", "translation": "czerwiec"}],
    "July":      [{"eng_lemma": "July",      "pos": "NOUN", "translation": "lipiec"}],
    "August":    [{"eng_lemma": "August",    "pos": "NOUN", "translation": "sierpień"}],
    "September": [{"eng_lemma": "September", "pos": "NOUN", "translation": "wrzesień"}],
    "October":   [{"eng_lemma": "October",   "pos": "NOUN", "translation": "październik"}],
    "November":  [{"eng_lemma": "November",  "pos": "NOUN", "translation": "listopad"}],
    "December":  [{"eng_lemma": "December",  "pos": "NOUN", "translation": "grudzień"}],
}

# Force calendar translations
def force_calendar_translations(row) -> dict:
    target = row["target"]
    if target in CALENDAR_PL:
        return {target: CALENDAR_PL[target]}
    return row["diki_translations_raw"]

df["diki_translations_raw"] = df.apply(force_calendar_translations, axis=1)

#### Find supported translations

Intersection between duolingo translations lemmas and dictionary translations (assumed to be in lemma form)

In [ ]:
# %% ── STAGE 3: FUNCTIONS ─────────────────────────────────────────────────────  

def build_inferred_lemmas(translations_str: str) -> list[dict]:
    """
    Morphologically analyse duolingo translations.
    Returns list of {'translation', 'morf_tag', 'pos', 'lemma'}.
    Skips phrase tokens (inner whitespace).
    """
    _RE_SIE_INFERRED = re.compile(r'\bsię\b')
    result = []
    for token in [t.strip() for t in translations_str.split(',') if t.strip()]:
        token = _RE_SIE_INFERRED.sub('', token).strip()
        if not token:
            continue
        if ' ' in token:
            continue
        for rec in morf_analyse_clean(token):
            result.append({
                'translation': token,
                'morf_tag':    rec['morf_tag'],
                'pos':         rec['pos'],
                'lemma':       rec['lemma'],
            })
    return result


def build_dictionary_lemmas(diki_translations: dict) -> list[dict]:
    """
    Flatten diki translations to {'eng_lemma', 'pos', 'lemma'}.
    """
    result = []
    for eng_lemma, trans_list in diki_translations.items():
        for t in trans_list:
            result.append({
                'eng_lemma': eng_lemma,
                'pos':       t['pos'],
                'lemma':     t['translation'],
            })
    return result


def build_dictionary_lemmas_analysed(diki_translations: dict) -> list[dict]:
    """
    Flatten + morphologically analyse diki translations.
    Use story:
    'If an valid dictionary translation of eng_word is (pos, lemma_translation),
    then given peculiar polish morphology,
    (pos, lemma_translation) can also be (pos', lemma_translation', support_morf_tag)'

    Returns {'eng_lemma', 'diki_pos', 'diki_translation', 'morf_tag', 'pos', 'lemma'}.
    """
    result = []
    for eng_lemma, trans_list in diki_translations.items():
        for t in trans_list:
            for rec in morf_analyse_clean(t['translation']):
                # if t['pos'] != rec['pos']:
                #     continue
                result.append({
                    'eng_lemma':        eng_lemma,
                    # 'diki_pos':         t['pos'],
                    # 'diki_translation': t['translation'],
                    'pos':              rec['pos'],
                    'lemma':            rec['lemma'],
                    'dict_morf_tag': rec['morf_tag'],
                })
    return result


def index_by_pos_lemma(entries: list[dict], index_keys: list[str]=['pos','lemma']) -> dict[tuple, list[dict]]:
    """Index list of dicts by (pos, lemma) key."""
    indexed: dict[tuple, list] = defaultdict(list)
    for entry in entries:
        key = tuple(entry[i] for i in index_keys)
        indexed[key].append(entry)
    return dict(indexed)


def build_supported_lemmas(
    inferred_indexed:   dict[tuple, list[dict]],
    dictionary_indexed: dict[tuple, list[dict]],
    drop_noncore: bool = True
) -> list[dict]:
    """
    Return dictionary_lemmas entries whose (pos, lemma) key exists in inferred_lemmas.
    Drop entries whos POS does not exist in CORE_POS {NOUN,ADJ,VERB}.
    Attach matching inferred forms as 'inferred_forms' for future tag-matching use.
    """
    common_keys = inferred_indexed.keys() & dictionary_indexed.keys()
    
    if drop_noncore:
        common_keys = {k for k in common_keys if k[0] in CORE_POS}
    result = []

    for k in common_keys:
        for dict_entry in dictionary_indexed[k]:
            for inferred_entry in inferred_indexed[k]:
                result.append({**dict_entry, **inferred_entry})
    return result


In [ ]:
# %% ── STAGE 3: RUN ─────────────────────────────────────────────────────

df["inferred_lemmas"] = df.apply(
    lambda row: build_inferred_lemmas(row["duolingo_translations"])
    if WORDS_TO_TRANSLATE_MASK[row.name] else {},
    axis=1,
)

df["dictionary_lemmas"] = df["diki_translations_raw"].apply(
    lambda raw: build_dictionary_lemmas(raw)
    if isinstance(raw, dict) and raw else {}
)

df["dictionary_lemmas_analysed"] = df["diki_translations_raw"].apply(
    lambda raw: build_dictionary_lemmas_analysed(raw)
    if isinstance(raw, dict) and raw else {}
)

# Build indexed columns
df["inferred_lemmas_indexed"] = df["inferred_lemmas"].apply(index_by_pos_lemma)
df["dictionary_lemmas_indexed"] = df["dictionary_lemmas"].apply(index_by_pos_lemma)
df["dictionary_lemmas_analysed_indexed"] = df["dictionary_lemmas_analysed"].apply(index_by_pos_lemma)


df["supported_lemmas"] = df.apply(
    lambda row: build_supported_lemmas(row["inferred_lemmas_indexed"], row["dictionary_lemmas_indexed"]),
    axis=1,
)


df["supported_lemmas_analysed"] = df.apply(
    lambda row: build_supported_lemmas(row["inferred_lemmas_indexed"], row["dictionary_lemmas_analysed_indexed"]),
    axis=1,
)

df["supported_translations"] = df["supported_lemmas"]

mask = df["supported_lemmas"].apply(lambda x: len(x) == 0)
df.loc[mask, "supported_translations"] = df.loc[mask, "supported_lemmas_analysed"]

df["supported_translations_indexed"] = df["supported_translations"].apply(index_by_pos_lemma)

#### Reevaluate eng_analyses

(possible pos, lemma of target words) based on supported translations

In [ ]:
def build_eng_analyses_supported(row) -> list[dict]:
    """
    Keep only eng_analyses entries whose (pos, lemma) exists in:
      1. supported_lemmas          (primary)
      2. supported_lemmas_analysed (fallback if 1 is empty)
      ^ combined in supported_transaltions
      3. dictionary_lemmas         (fallback if 2 is empty)
      4. []                        (final fallback)
    """
    supported     = row["supported_translations"]
    dictionary    = row["dictionary_lemmas"]

    if supported:
        reference = {(e["pos"], e["eng_lemma"]) for e in supported}
    elif dictionary:
        reference = {(e["pos"], e["eng_lemma"]) for e in dictionary}
    else:
        return []
    
    return [
        a for a in row["eng_analyses"]
        if (a["pos"], a["eng_lemma"]) in reference
    ]

df["_eng_analyses"] = df["eng_analyses"].copy()
df["eng_analyses"] = df.apply(build_eng_analyses_supported, axis=1)

df["_eng_pos_set"] = df["eng_pos_set"].copy()
df["eng_pos_set"] = df["eng_analyses"].apply(
    lambda a: frozenset(x["pos"] for x in a)
)

df["_eng_word_type"] = df["eng_word_type"].copy()
df["eng_word_type"] = df.apply(
    lambda row: classify_word_type(row["target"], row["eng_pos_set"]), axis=1
)

df["_eng_lexeme_form"] = df["eng_lexeme_form"].copy()
df["eng_lexeme_form"] = df.apply(
    lambda row: classify_lexeme_form(row["target"], row["eng_analyses"]), axis=1
)

df["_eng_lemma_pos"] = df["eng_lemma_pos"].copy()
df["eng_lemma_pos"] = df.apply(
    lambda row: build_lemma_pos_dict(row["target"], row["eng_analyses"]), axis=1
)

In [ ]:
# _ALL_TARGETS= df["target"].to_list()

# def get_learned_inflections(eng_lemma_pos: dict, target: str) -> list[str]:
#     """
#     Generate all possible inflections for each lemma in eng_lemma_pos,
#     return only those forms that already exist as target words in the table.
#     """
#     found: set[str] = set()
#     for lemma, pos_set in eng_lemma_pos.items():
#         for pos in pos_set:
#             inflections = getAllInflections_better(lemma, upos=pos)
#             for forms in inflections.values():
#                 for form in forms:
#                     if (form in _ALL_TARGETS) and (form != lemma) and (form != target):
#                         found.add(form)
#     return sorted(found)

# df["eng_learned_inflections"] = df.apply(lambda row: get_learned_inflections(row["eng_lemma_pos"], row["target"]), axis=1)

#### Stats

In [ ]:
active = WORDS_TO_TRANSLATE_MASK

print("=== supported_lemmas distribution ===")
sup_counts = df[active]["supported_lemmas"].apply(len)
print(sup_counts.describe().round(2))
print(f"Rows with 0 supported lemmas : {(sup_counts == 0).sum()}")
print(f"Rows with ≥1 supported lemmas: {(sup_counts  > 0).sum()}")

print("\n=== before - eng_analyses distribution ===")
ana_counts = df[active]["_eng_analyses"].apply(len)
print(ana_counts.describe().round(2))
print(f"Rows with 0 eng_analyses: {(ana_counts == 0).sum()}")

print("\n=== after - eng_analyses distribution ===")
ana_counts = df[active]["eng_analyses"].apply(len)
print(ana_counts.describe().round(2))
print(f"Rows with 0 eng_analyses: {(ana_counts == 0).sum()}")

# print("\n=== eng_learned_inflections — coverage ===")
# inf_counts = df["eng_learned_inflections"].apply(len)
# print(f"Rows with ≥1 learned inflection in table: {(inf_counts > 0).sum()}")
# print(inf_counts[inf_counts > 0].describe().round(2))

print("\n=== blacklisted words zeroed out ===")
blacklisted_mask = df['is_blacklisted']==True
print(f"Blacklisted rows: {blacklisted_mask.sum()}")
print(df[blacklisted_mask]["target"].tolist())

print("\n=== calendar overrides applied ===")
calendar_mask = df["target"].str.strip().isin(CALENDAR_PL)
print(f"Calendar rows: {calendar_mask.sum()}")
print(df[calendar_mask]["target"].tolist())

In [ ]:
# ── Atomic conditions ─────────────────────────────────────────────────────────
# has_supported_lemma = df['supported_lemmas'].apply(lambda x: isinstance(x, list) and len(x) > 0)
# has_supported_lemma_analysed = df['supported_lemmas_analysed'].apply(lambda x: isinstance(x, list) and len(x) > 0)
has_supported = df['supported_translations'].apply(lambda x: isinstance(x, list) and len(x) > 0)
eng_phrase = df['eng_word_type'] == 'phrase'
t_phrase = df['t_word_type'] == 'phrase'
is_blacklisted = df['is_blacklisted'] == True
is_function = df['eng_word_type'].isin(['punctuation', 'function'])

# ── Final mask ────────────────────────────────────────────────────────────────
final = (has_supported | eng_phrase | t_phrase) & ~is_function & ~is_blacklisted

#### Exploration, extra stats

In [ ]:
# TODO: 
# for core inflected - different strategy - scan targets for their existing inflections. to inflection asing the translation from target and expand it only to appropriate forms


# check target tag and allow
# if diki NOUN - allow for matching with morfeusz VERB ger
# if diki ADJ - allow for matching with morfeusz VERB ppas (wyschnięty, przesuszony)
# if diki ADJ - allow for matching with morfeusz VERB pact (niepijący, trwający)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

pd.reset_option('display.max_rows')
# pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')

In [ ]:
c = df[~is_blacklisted][~(has_supported)][~eng_phrase][t_phrase]
d = df[~is_blacklisted][(has_supported) | eng_phrase]# | c]
# remove type punctuation, function
# allow core, core/function, other
print(len(d))
d[['eng_word_type','eng_lexeme_form', 't_word_type']].value_counts(dropna=False)
# d[(d['eng_word_type']=='core') & (d['eng_lexeme_form'])]

In [ ]:
tmp = df.copy()

found = (tmp['supported_lemmas'].apply(lambda x: isinstance(x, list) and len(x) > 0))
found_onlystar = (~found & (tmp['supported_lemmas_analysed'].apply(lambda x: isinstance(x, list) and len(x) > 0)))
found_star = (found | (tmp['supported_lemmas_analysed'].apply(lambda x: isinstance(x, list) and len(x) > 0)))
phrase = tmp['eng_word_type']=='phrase'
t_phrase = tmp['t_word_type']=='phrase'
final = ((WORDS_TO_TRANSLATE_MASK & found) | phrase)

print(f"total len: {len(tmp)}")
print(f"words queried for translation: {len(tmp[WORDS_TO_TRANSLATE_MASK])}")
print(f"words with supported translations: {len(tmp[found])}")
print(f"words with any supported translations*: {len(tmp[found_star])}")
print(f"words with supported translations (only star): {len(tmp[found_onlystar])}")
print(f"words queried and found: {len(tmp[WORDS_TO_TRANSLATE_MASK & found])}")
print(f"words queried but not found: {len(tmp[WORDS_TO_TRANSLATE_MASK & ~found])}")
# print(f"words queried but not found (exclude if any translation is a phrase - will case mismatch)): {len(tmp[WORDS_TO_TRANSLATE_MASK & ~found & ~t_anyphrase])}")
print(f"words queried but not found (exclude if all translations are a phrase - will case mismatch)): {len(tmp[WORDS_TO_TRANSLATE_MASK & ~found & ~t_phrase])}")
print(f"words phrases: {len(tmp[phrase])}")
print(f"words for substitution: queried and found, or phrase: {len(tmp[(WORDS_TO_TRANSLATE_MASK & found) | phrase])}")
print(f"discarded/unmatched words: {len(tmp) - len(tmp[(WORDS_TO_TRANSLATE_MASK & found) | phrase])}")

tmp['diki_trans'] = tmp['dictionary_lemmas_indexed'].apply(lambda x: [k for k in x.keys()])
tmp['diki_trans_star'] = tmp['dictionary_lemmas_analysed_indexed'].apply(lambda x: [k for k in x.keys()])
tmp['duo_trans'] = tmp['inferred_lemmas_indexed'].apply(lambda x: [k for k in x.keys()])
tmp['sup_trans'] = tmp['supported_translations_indexed'].apply(lambda x: [k for k in x.keys()])


trash = tmp[~final]
trash[['eng_word_type','eng_lexeme_form', 't_word_type']].value_counts(dropna=False)


### Stage 4

In [ ]:
# Setup
df = df[final].copy()
has_supported = df['supported_translations'].apply(lambda x: isinstance(x, list) and len(x) > 0)

##### duplicate stats

In [ ]:
# Duplicate stats for (pos, lemma)
from collections import defaultdict
tmp = df[df['eng_lexeme_form'].isin(['lemma', 'OTHER'])]
# Zbierz wszystkie (pos, pol_lemma) → [target words]
_lemma_to_words: dict[tuple, list[str]] = defaultdict(set)

for idx in tmp.index:
    sup = tmp.at[idx, 'supported_translations']
    if not isinstance(sup, list):
        continue
    for entry in sup:
        key = (entry.get('pos'), entry.get('lemma'))
        _lemma_to_words[key].add(tmp.at[idx, 'target'])

_shared = {k: v for k, v in _lemma_to_words.items() if len(v) > 1}
print(f"only lemma + OTHER, no inflections et.al")
print(f"Unique (pos, pol_lemma) pairs: {len(_lemma_to_words)}")
print(f"Pairs shared by >1 word:       {len(_shared)}")
print(f"Affected target words:          {len({w for words in _shared.values() for w in words})}")
print()
for (pos, lemma), words in sorted(_shared.items(), key=lambda x: -len(x[1]))[:10]:
    print(f"  ({pos:5s}, {lemma:20s}) → {words}")

#### 1. Lemma-level deduplication

##### sorting

In [ ]:
from pathlib import Path

def load_freq_list(path: str) -> list[str]:
    """One word per line, already sorted most→least frequent."""
    return [
        w.strip().lower()
        for w in Path(path).read_text(encoding="utf-8").splitlines()
        if w.strip()
    ]

def build_freq_lookup(*freq_list_files: str) -> dict[str, int]:
    # Combine into a single word → rank lookup.
    # If a word appears in multiple lists (e.g. "hope" is both noun and verb),
    # keep the best (lowest / most frequent) rank across all lists.
    freq_lookup: dict[str, int] = {}

    for filename in freq_list_files:
        freq_list = load_freq_list(filename)
        for rank, word in enumerate(freq_list):
            if word not in freq_lookup or rank < freq_lookup[word]:
                freq_lookup[word] = rank

    return freq_lookup


def lemma_criticality(row) -> int:
    """
    1 = target depends on exactly one (pos, lemma) pair — losing it orphans the target.
    0 = target has >1 lemma, can afford to lose one.
    """
    supported = row.get("supported_translations") or []
    unique_lemmas = {(e["pos"], e["lemma"]) for e in supported}
    return 1 if len(unique_lemmas) == 1 else 0


# Dictionary ambiguity penalty — from diki_cache
# higher = fewer senses = more "precise" claim
def diki_sense_count(eng_lemma: str) -> int:
    entries = diki_cache.get(eng_lemma, [])
    return len(entries)

def diki_sense_count_for_row(row) -> int:
    lemma_pos = row.get("eng_lemma_pos") or {}
    if not lemma_pos:
        return diki_sense_count(row["target"])  # phrase/fallback case
    counts = [len(diki_cache.get(lemma, [])) for lemma in lemma_pos]
    return min(counts) if counts else 0


DEDUP_PRIORITY = [
    # (eng_word_type,   lexeme_form,                  t_word_type)  — wyższy index = wyższa ochrona
    ("phrase",          None,                          None),         # 0 — najniższa
    (None,              {"ambiguous","multi_inflected"},None),        # 1
    ("OTHER",           None,                          "not_core"),   # 2
    ("OTHER",           None,                          "core"),       # 3
    ("core/function",   "not_lemma",                   None),         # 4
    ("core/function",   "lemma",                       None),         # 5
    ("core",            "not_lemma",                   None),         # 6
    ("core",            "lemma",                       None),         # 7 — najwyższa
]

def priority_rank(eng_word_type: str, lexeme_form: str, t_word_type: str) -> int:
    best = -1
    for idx, (want_ewt, want_lex, want_twt) in enumerate(DEDUP_PRIORITY):
        if want_ewt is not None and want_ewt != eng_word_type:
            continue
        if want_lex is not None:
            if isinstance(want_lex, set):
                if lexeme_form not in want_lex:
                    continue
            elif want_lex == "lemma" and lexeme_form != "lemma":
                continue
            elif want_lex == "not_lemma" and lexeme_form == "lemma":
                continue
        if want_twt is not None:
            is_core = (t_word_type == "core")
            if want_twt == "core" and not is_core:
                continue
            if want_twt == "not_core" and is_core:
                continue
        best = max(best, idx)
    return best if best >= 0 else 0

In [ ]:
freq_lookup = build_freq_lookup(
    "top_english_nouns_lower_50000.txt",
    "top_english_verbs_lower_50000.txt",
    "top_english_adjs_lower_50000.txt",
)
df["_freq_rank"] = df["target"].apply(lambda w: freq_lookup.get(w.lower(), 999_999))

df["_lemma_criticality"] = df.apply(lemma_criticality, axis=1)

df["_priority"] = df.apply(lambda r: priority_rank(r["eng_word_type"], r["eng_lexeme_form"], r["t_word_type"]), axis=1)

df["_lemma_specificity"] = df["supported_translations"].apply(lambda s: 1 / max(len(s or []), 1))

##### deduplication

In [ ]:
df_sorted = df.sort_values(
    ["_lemma_criticality", "_priority", "_freq_rank", "_lemma_specificity", "target"],
    ascending=[False, False, True, False, True],
).copy()

compete_mask = has_supported & df_sorted["eng_lexeme_form"].isin(["lemma", "OTHER"])

lemma_candidates = defaultdict(list)
for _, row in df_sorted[compete_mask].iterrows():
    for entry in (row.get("supported_translations") or []):
        lemma_candidates[(entry["pos"], entry["lemma"])].append(row["target"])


lemma_winner = {}
lemma_dedup_log = []
for _, row in df_sorted[compete_mask].iterrows():
    target = row["target"]
    row_lemmas = {(e["pos"], e["lemma"]) for e in (row.get("supported_translations") or [])}
    for key in row_lemmas:
        if key not in lemma_winner:
            lemma_winner[key] = target
            if len(set(lemma_candidates[key])) > 1:
                lemma_dedup_log.append({"pos_lemma": key,
                    "candidates": sorted(set(lemma_candidates[key])), "winner": target})

lemma_dedup_log_df = pd.DataFrame(lemma_dedup_log)

In [ ]:
def filter_supported(row):
    if row["eng_lexeme_form"] not in ("lemma", "OTHER"):
        return row["supported_translations"]  # inflected/ambiguous/multi — untouched
    target = row["target"]
    return [e for e in (row.get("supported_translations") or [])
            if lemma_winner.get((e["pos"], e["lemma"]), target) == target]

df_sorted["supported_translations_dedup"] = df_sorted.apply(filter_supported, axis=1)

#### 2. Generate inflections

In [ ]:
df_sorted["supported_translations_dedup_eng_indexed"] = df_sorted["supported_translations_dedup"].apply(lambda x: index_by_pos_lemma(x, ['pos', 'eng_lemma']))
df_sorted['eng_analyses_indexed'] = df_sorted["eng_analyses"].apply(lambda x: index_by_pos_lemma(x, ['pos', 'eng_lemma']))

df_sorted["inflection_generation_units"] = df_sorted.apply(
    lambda row: build_supported_lemmas(row["supported_translations_dedup_eng_indexed"], row["eng_analyses_indexed"]),
    axis=1,
)

# Deduplicate
df_sorted["inflection_generation_units"] = df_sorted["inflection_generation_units"].apply(
    lambda lst: list({
        tuple(sorted(d.items())): d for d in lst
        }.values())
)

# Confirm deduplication
d1 = df_sorted["inflection_generation_units"].apply(len)
d2 = df_sorted["inflection_generation_units"].apply(lambda lst: len({tuple(sorted(d.items())) for d in lst}))
assert (d1 == d2).all()

##### mapping

In [ ]:
# ── Tabela 1: morf_pos → metadane ────────────────────────────────────────────
MORF_POS_META = [
    # morf_pos   eng_name                          info                                          penn_tags
    ("subst",   "noun",                            "inflects: number, case, gender",             ["NN","NNS","NNP","NNPS"]),
    ("depr",    "depreciative noun",               "like subst, only pl:nom.acc.voc:m2",         ["NN","NNS"]),
    ("num",     "numeral",                         "inflects: case, gender, collectivity",       ["NN","NNS"]),
    ("numcol",  "collective numeral",              "like num",                                   ["NN","NNS"]),
    ("adj",     "adjective",                       "inflects: number, case, gender, degree",     ["JJ","JJR","JJS"]),
    ("adja",    "ad-adjectival adjective",         "indeclinable, no features",                  ["JJ"]),
    ("adjp",    "post-prepositional adjective",    "only gen/dat case forms",                    ["JJ"]),
    ("adjc",    "predicative adjective",           "like adj",                                   ["JJ","JJR","JJS"]),
    ("fin",     "non-past finite verb",            "inflects: number, person, aspect",           ["VB","VBP","VBZ"]),
    ("praet",   "past tense / l-participle",       "inflects: number, gender, person, aspect",   ["VBD"]),
    ("inf",     "infinitive",                      "inflects: aspect only",                      ["VB"]),
    ("impt",    "imperative",                      "inflects: number, person, aspect",           ["VB"]),
    ("imps",    "impersonal",                      "inflects: aspect only",                      ["VBN"]),
    ("ger",     "gerund / verbal noun",            "inflects: number, case; always neuter",      ["VBG","NN"]),
    ("pact",    "active adjectival participle",    "inflects like adj; imperf only",             ["VBG","JJ"]),
    ("pacta",   "indeclinable active participle",  "no features",                                ["VBG"]),
    ("pcon",    "contemporary adv. participle",    "inflects: aspect only (imperf)",             ["VBG"]),
    ("pant",    "anterior adv. participle",        "inflects: aspect only (perf)",               ["VBN"]),
    ("ppas",    "passive adjectival participle",   "inflects like adj; imperf+perf",             ["VBN","JJ"]),
]

In [ ]:
_RULES_RAW = [
    # penn_tag   morf_pos   require_features (AND)      forbid_features (OR)
    # ADJ
    ("JJ",   "adj",    ["pos"],            []                   ),
    ("JJ",   "adja",   [],                 []                   ),
    ("JJ",   "adjc",   ["pos"],            []                   ),
    ("JJ",   "adjp",   [],                 []                   ),
    ("JJ",   "ppas",   [],                 []                   ),
    ("JJ",   "pact",   [],                 []                   ),
    ("JJR",  "adj",    ["com"],            []                   ),
    ("JJR",  "adjc",   ["com"],            []                   ),
    ("JJS",  "adj",    ["sup"],            []                   ),
    ("JJS",  "adjc",   ["sup"],            []                   ),
    # NOUN — NN singular, NNS plural
    ("NN",   "subst",  ["sg"],             []                   ),
    ("NN",   "depr",   ["sg"],             []                   ),
    ("NN",   "num",    [],                 []                   ),
    ("NN",   "numcol", [],                 []                   ),
    ("NN",   "ger",    ["sg"],             []                   ),
    ("NNP",  "subst",  ["sg"],             []                   ),
    ("NNPS", "subst",  ["pl"],             []                   ),
    ("NNS",  "subst",  ["pl"],             []                   ),
    ("NNS",  "depr",   ["pl"],             []                   ),
    ("NNS",  "num",    [],                 []                   ),
    ("NNS",  "numcol", [],                 []                   ),
    # VERB
    ("VB",   "fin",    [],                 [("sg","ter")]       ),
    ("VB",   "impt",   [],                 []                   ),
    ("VB",   "inf",    [],                 []                   ),
    ("VBD",  "praet",  [],                 []                   ),
    ("VBG",  "pact",   [],                 []                   ),
    ("VBG",  "pacta",  [],                 []                   ),
    ("VBG",  "pcon",   [],                 []                   ),
    ("VBG",  "ger",    ["sg"],             []                   ),
    ("VBN",  "imps",   [],                 []                   ),
    ("VBN",  "pant",   [],                 []                   ),
    ("VBN",  "ppas",   [],                 []                   ),
    ("VBP",  "fin",    [],                 [("sg","ter")]              ),
    ("VBP",  "impt",   [],                 []                   ),
    ("VBP",  "inf",    [],                 []                   ),
    ("VBZ",  "fin",    ["ter", "sg"],      []                   ),
]

# (penn_tag, morf_pos) → {require, forbid}
GENERATION_LOOKUP: dict[tuple[str, str], dict] = {
    (penn, m_pos): {"require": req, "forbid": forb}
    for penn, m_pos, req, forb in _RULES_RAW
}

##### generation

In [ ]:
# ── 4.0  Tag parsing utilities ───────────────────────────────────────────────

def parse_morf_tag(tag: str) -> list[set[str]]:
    """
    'pact:sg:nom.voc:m1.m2.m3:imperf:neg'
    → [{'pact'}, {'sg'}, {'nom','voc'}, {'m1','m2','m3'}, {'imperf'}, {'neg'}]
    """
    return [set(cat.split('.')) for cat in tag.split(':')]

def has_tag_value(parts: list[set[str]], value) -> bool:
    """True if `value` appears in any category of the parsed tag.
    If `value` is a tuple, True only if ALL elements appear (AND)."""
    if isinstance(value, tuple):
        return all(has_tag_value(parts, v) for v in value)
    return any(value in cat for cat in parts)

def tag_class(parts: list[set[str]]) -> str:
    """Morphological class — always the first category, always a single value."""
    return next(iter(parts[0])) if parts else ''

In [ ]:
# ── 4.2  Form filter ──────────────────────────────────────────────────────────

def _tag_passes_constraint(morf_tag: str, penn_tag: str, allow_neg: bool = True) -> bool:
    """
    Check whether the inferred (confirmed) morf_tag satisfies
    the generation constraint for this penn_tag.
    If not — generation is not permitted at all for this unit.
    """
    parts    = parse_morf_tag(morf_tag)
    morf_pos = tag_class(parts)
    is_pt    = has_tag_value(parts, "pt")  # plurale tantum override

    constraint = GENERATION_LOOKUP.get((penn_tag, morf_pos))
    if constraint is None:
        return False

    for req in constraint["require"]:
        if req == "sg" and is_pt:
            continue  # plurale tantum has no singular — don't require it
        if not has_tag_value(parts, req):
            return False
    for forb in constraint["forbid"]:
        if has_tag_value(parts, forb):
            return False
    if not allow_neg and has_tag_value(parts, "neg"):
        return False
    return True


# ── 4.3  Row-level generation ─────────────────────────────────────────────────
def morf_generate_clean(word: str) -> list[tuple[str,str]]:
    results = []

    for orth, lemma_raw, tag, features, flags in morf.generate(word):
        # flags may be a single string in a list
        if len(flags)==1:
            flags = flags[0].split(',')

        if tag == 'ign':
            continue
        feat_set = set(features) if features else set()
        flag_set = set(flags)    if flags    else set()
        if feat_set & MORF_BANNED_FEATURES:
            continue
        if flag_set & MORF_BANNED_QUALIFIERS:
            continue

        results.append((orth, tag))

    return results


def generate_forms_for_row(units) -> list[dict]:
    """
    For each row, group inflection_generation_units by pol_lemma,
    call morf.generate() once per lemma, filter all generated forms
    against all units for that lemma, return list of result dicts.
    Empty forms → empty list (no fallback to lemma).
    """
    if not units:
        return []

    # Group units by pol_lemma
    by_lemma: dict[str, list[dict]] = defaultdict(list)
    for u in units:
        by_lemma[u["lemma"]].append(u)

    result = []
    for lemma, lemma_units in by_lemma.items():

        try:
            generated = morf_generate_clean(lemma)
        except Exception as e:
            print(e)
            generated = []

        forms: set[str] = set()

        for generated_surface, generated_morf_tag in generated:
            for u in lemma_units:
                eng_penn_tag    = u["penn_tag"]
                translation_morf_tag = u["morf_tag"]

                # Check if the duolingo translation fulfills constraints. If the constraints are supported by a known translation.
                if not _tag_passes_constraint(translation_morf_tag, eng_penn_tag):
                    continue

                # Check if generated form fulfills constraints. 
                translation_is_neg = has_tag_value(parse_morf_tag(translation_morf_tag), "neg")
                if _tag_passes_constraint(generated_morf_tag, eng_penn_tag, translation_is_neg):
                    forms.add(generated_surface)
                    break

        result.append({
            "lemma":     lemma,
            "eng_lemma": lemma_units[0]["eng_lemma"],
            "pos":       lemma_units[0]["pos"],
            "penn_tags": sorted({u["penn_tag"] for u in lemma_units}),
            "morf_tags": sorted({u["morf_tag"] for u in lemma_units}),
            "forms":     sorted(forms),   # empty list if no forms passed filter
        })

    return result


def build_all_translations(row) -> set[str]:
    result = set()
    for entry in (row.get("generated_translations") or []):
        result.update(entry.get("forms", []))

    had_original_support = bool(row.get("supported_translations"))
    if not result and not had_original_support:
        for t in str(row.get("duolingo_translations", "")).split(","):
            if t.strip():
                result.add(t.strip())
    return result

In [ ]:
# ── 4.4  Apply ────────────────────────────────────────────────────────────────
df_sorted["generated_translations"] = df_sorted["inflection_generation_units"].apply(generate_forms_for_row)
df_sorted["all_translations"] = df_sorted.apply(build_all_translations, axis=1)

##### duplicate stats

In [ ]:
from collections import defaultdict

# tmp = df_sorted[compete_mask]
tmp = df_sorted

# Zbierz wszystkie (pos, pol_lemma) → [target words]
_inflections_to_words: dict[str, list[str]] = defaultdict(set)

for idx in tmp.index:
    sup = tmp.at[idx, 'all_translations']
    for t in sup:
        _inflections_to_words[t].add(tmp.at[idx, 'target'])

_shared = {k: v for k, v in _inflections_to_words.items() if len(v) > 1}
print(f"Unique inflections: \t\t{len(_inflections_to_words)}")
print(f"Inflections shared by >1 word: \t{len(_shared)}")
print(f"Affected target words: \t\t{len({w for words in _shared.values() for w in words})}")
print()
for lemma, words in sorted(_shared.items(), key=lambda x: -len(x[1]))[:10]:
    print(f"  {lemma:10s} → {words}")

#### 3. Deduplicate inflections

##### sorting

In [ ]:
df_sorted["_form_criticality"] = ((df_sorted["all_translations"].apply(len) == 1) & (df_sorted["eng_word_type"] != "phrase")).astype(int)

df_sorted["_form_specificity"] = df_sorted["all_translations"].apply(lambda s: 1 / max(len(s), 1))

##### deduplication

In [ ]:
df_sorted = df_sorted.sort_values(
    ["_form_criticality", "_priority", "_freq_rank", "_form_specificity", "target"],
    ascending=[False, False, True, False, True],
).copy()


candidates: dict[str, list[str]] = defaultdict(list)
for _, row in df_sorted.iterrows():
    for pol_word in row["all_translations"]:
        candidates[pol_word].append(row["target"])


pol_to_eng: dict[str, str] = {}
dedup_log: list[dict] = []

for _, row in df_sorted.iterrows():
    target = row["target"]
    for pol_word in row["all_translations"]:
        if pol_word not in pol_to_eng:
            pol_to_eng[pol_word] = target
            if len(candidates[pol_word]) > 1:
                dedup_log.append({
                    "pol_word":    pol_word,
                    "candidates":  candidates[pol_word],
                    "winner":      target,
                })

inflection_dedup_log_df = pd.DataFrame(dedup_log)

In [ ]:
def apply_final_dedup(row):
    target = row["target"]
    return {w for w in row["all_translations"] if pol_to_eng.get(w) == target}

df_sorted["all_translations_dedup"] = df_sorted.apply(apply_final_dedup, axis=1)

##### dropped rows - stats

In [ ]:
orphaned = df_sorted[df_sorted["all_translations_dedup"].apply(len) == 0]
print(f"Orphaned targets (with 0 translations after dedup): {len(orphaned)}")
print(orphaned["target"].tolist())

### Stage 5 - stats

In [ ]:
# Straty na każdym etapie pipeline'u
print("=" * 70)
print("1. ATTRITION ACROSS PIPELINE STAGES")
print("=" * 70)

n_raw = len(df_raw)                       # before Stage 1 (all duolingo rows)
n_stage3_final = len(df[final])           # entering Stage 4 (has_supported | phrase, not blacklisted)
n_after_lemma_dedup = (df_sorted["supported_translations_dedup"].apply(len) > 0).sum() \
                       + (~df_sorted["eng_lexeme_form"].isin(["lemma", "OTHER"])).sum()
n_after_inflection_dedup = (df_sorted["all_translations_dedup"].apply(len) > 0).sum()

print(f"Raw duolingo rows:                          {n_raw}")
print(f"Entering Stage 4 (after Stage 3 filter):     {n_stage3_final}")
print(f"With ≥1 translation after LEMMA dedup:       {n_after_lemma_dedup}  "
      f"(lost {n_stage3_final - n_after_lemma_dedup})")
print(f"With ≥1 translation after INFLECTION dedup:  {n_after_inflection_dedup}  "
      f"(lost {n_after_lemma_dedup - n_after_inflection_dedup} more)")
print(f"Total lost vs. Stage 3 output:                {n_stage3_final - n_after_inflection_dedup} "
      f"({(n_stage3_final - n_after_inflection_dedup)/n_stage3_final:.1%})")

In [ ]:
# Expansion factor (duolingo raw → final inflected forms)
print("=" * 70)
print("2. TRANSLATION EXPANSION FACTOR")
print("=" * 70)

def count_raw_duo(s):
    return len([t for t in str(s).split(",") if t.strip()])

df_sorted["_n_duo_raw"] = df_sorted["duolingo_translations"].apply(count_raw_duo)
df_sorted["_n_final"]   = df_sorted["all_translations_dedup"].apply(len)

survived = df_sorted[df_sorted["_n_final"] > 0]

total_raw   = survived["_n_duo_raw"].sum()
total_final = survived["_n_final"].sum()

print(f"Total raw duolingo translation tokens (survived rows): {total_raw}")
print(f"Total final deduplicated translations:                 {total_final}")
print(f"Overall expansion factor:                               {total_final/total_raw:.2f}x")
print(f"Avg raw translations / word:                            {survived['_n_duo_raw'].mean():.2f}")
print(f"Avg final translations / word:                          {survived['_n_final'].mean():.2f}")
print()
print(survived["_n_final"].describe().round(2))

In [ ]:
# Core vs. phrase vs. other breakdown
print("=" * 70)
print("3. CORE / PHRASE / OTHER BREAKDOWN (survived rows)")
print("=" * 70)

survived = df_sorted[df_sorted["all_translations_dedup"].apply(len) > 0]

print("By eng_word_type:")
print(survived["eng_word_type"].value_counts(dropna=False))
print()
print("By eng_lexeme_form:")
print(survived["eng_lexeme_form"].value_counts(dropna=False))
print()

is_core_with_inflections = survived["eng_word_type"].isin(["core", "core/function"]) & \
                           (survived["inflection_generation_units"].apply(len) > 0)
print(f"Core words WITH supported translation + inflections: {is_core_with_inflections.sum()}")
print(f"Phrases / OTHER (fallback, no inflection generation):  "
      f"{(~is_core_with_inflections).sum()}")

In [ ]:
# Phrase contamination in all_translations_dedup
print("=" * 70)
print("4. PHRASE CONTAMINATION IN FINAL TRANSLATIONS")
print("=" * 70)

def split_phrase_flags(translations: set[str]) -> tuple[int, int]:
    n_phrase = sum(1 for t in translations if " " in t.strip())
    n_word   = len(translations) - n_phrase
    return n_phrase, n_word

flags = survived["all_translations_dedup"].apply(split_phrase_flags)
survived = survived.assign(
    _n_phrase_trans=flags.apply(lambda x: x[0]),
    _n_word_trans=flags.apply(lambda x: x[1]),
)

has_any_phrase   = survived["_n_phrase_trans"] > 0
has_only_phrases = (survived["_n_phrase_trans"] > 0) & (survived["_n_word_trans"] == 0)

print(f"Targets with ≥1 phrase in all_translations_dedup:  {has_any_phrase.sum()} "
      f"({has_any_phrase.mean():.1%})")
print(f"Targets with ONLY phrases (no single-word forms):  {has_only_phrases.sum()} "
      f"({has_only_phrases.mean():.1%})")
print()
print("Sample targets with only-phrase translations:")
print(survived[has_only_phrases][["target", "all_translations_dedup"]].head(50).to_string())

In [ ]:
# Manual inspection — words lost at each dedup stage
print("=" * 70)
print("5. WORDS LOST AT LEMMA-DEDUP STAGE — for manual review")
print("=" * 70)

lost_at_lemma = df_sorted[
    (df_sorted["supported_translations_dedup"].apply(len) == 0) &
    (df_sorted["eng_lexeme_form"].isin(["lemma", "OTHER"])) &
    (df_sorted["supported_translations"].apply(lambda x: isinstance(x, list) and len(x) > 0))
]
print(f"Count: {len(lost_at_lemma)}")
print(sorted(lost_at_lemma["target"].to_list()[:30]))
# review_lemma.to_csv("lost_at_lemma_dedup.csv", index=False)   # for full manual review

targets = pd.DataFrame(lost_at_lemma["target"].sort_values())
lost_words = lemma_dedup_log_df.copy()
lost_words = (
    lost_words
    .assign(candidates2=lambda d: d["candidates"])
    .explode("candidates2")
    .merge(
        targets.rename(columns={"target": "lost_word"}),
        left_on="candidates2",
        right_on="lost_word",
        how="right",
    )
    .drop(columns="candidates2")
)
print(lost_words[['lost_word', 'pos_lemma', 'candidates', 'winner']][:30])


# %%
print("=" * 70)
print("6. WORDS LOST AT INFLECTION-DEDUP STAGE — for manual review")
print("=" * 70)

lost_at_inflection = df_sorted[
    (df_sorted["all_translations_dedup"].apply(len) == 0) &
    (df_sorted["all_translations"].apply(len) > 0)
]
print(f"Count: {len(lost_at_inflection)}")
review_inflection = lost_at_inflection[["target", "eng_word_type", "eng_lexeme_form",
                                          "all_translations", "duolingo_translations"]]
print(sorted(lost_at_inflection['target'].to_list()[:30]))
# review_inflection.to_csv("lost_at_inflection_dedup.csv", index=False)


targets = pd.DataFrame(lost_at_inflection["target"].sort_values())
lost_words = inflection_dedup_log_df.copy()
lost_words = (
    lost_words
    .assign(candidates2=lambda d: d["candidates"])
    .explode("candidates2")
    .merge(
        targets.rename(columns={"target": "lost_word"}),
        left_on="candidates2",
        right_on="lost_word",
        how="right",
    )
    .drop(columns="candidates2")
)

print(lost_words[['lost_word', 'pol_word', 'candidates', 'winner']][:30])

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

pd.reset_option('display.max_rows')
# pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')

In [ ]:
lost_at_generation = df_sorted[
    (df_sorted["supported_translations_dedup"].apply(len) > 0) &
    (df_sorted["inflection_generation_units"].apply(len) > 0) &
    (df_sorted["all_translations"].apply(len) == 0)
]
print(f"Lost at generation stage (units existed, 0 forms passed constraints): {len(lost_at_generation)}")
print(lost_at_generation["target"].tolist())

In [ ]:
# %%
print("=" * 70)
print("X. WORDS LOST AT GENERATION STAGE — units existed, 0 forms passed")
print("=" * 70)

def summarize_failed_units(row):
    failures = []
    for u in row["inflection_generation_units"]:
        eng_lemma = u['eng_lemma']
        penn_tag = u["penn_tag"]
        morf_tag = u["morf_tag"]
        ok = _tag_passes_constraint(morf_tag, penn_tag)
        failures.append({
            "eng_lemma": eng_lemma,
            "penn_tag": penn_tag,
            "morf_tag": morf_tag,
            "morf_class": tag_class(parse_morf_tag(morf_tag)),
            "translation_ok": ok,
        })
    return failures

lost_at_generation = df_sorted[
    (df_sorted["supported_translations_dedup"].apply(len) > 0) &
    (df_sorted["inflection_generation_units"].apply(len) > 0) &
    (df_sorted["all_translations"].apply(len) == 0)
].copy()

lost_at_generation["_failed_units"] = lost_at_generation.apply(summarize_failed_units, axis=1)

print(f"Count: {len(lost_at_generation)}")
for _, row in lost_at_generation.iterrows():
    print(f"\n{row['target']}  (duolingo: {row['duolingo_translations']})")
    for f in row["_failed_units"]:
        status = "OK " if f["translation_ok"] else "FAIL"
        print(f"    [{status}] lemma={f['eng_lemma']} penn={f['penn_tag']:6s} morf_class={f['morf_class']:8s} morf_tag={f['morf_tag']}")

# lost_at_generation[["target","duolingo_translations","_failed_units"]].to_csv("lost_at_generation.csv", index=False)

In [ ]:
counter = 0
diffs = []
for t, g in merged_groups.groupby("target"):
    raw_sets = [set(x.strip() for x in s.split(",")) for s in g["duolingo_translations_raw"]]
    union = set.union(*raw_sets)
    # print(f"{t}: {[len(s) for s in raw_sets]} tokens per raw row → {len(union)} after merge")
    if max([len(s) for s in raw_sets]) < len(union):
        counter += 1
        diffs.append(len(union) - max([len(s) for s in raw_sets]))
print(pd.DataFrame(diffs).describe())
print("% cases where stripping added translations", (counter/len(merged_groups.groupby("target"))*100), "diff ", diffs)
# Jeśli union jest większy niż każdy pojedynczy zbiór osobno (np. [1, 1] → 2 zamiast [1,1]→1), 
# to merge faktycznie wzbogacił słownik kandydatów wejściowych do generacji 
# — to jest bezpośredni dowód korzyści, zanim jeszcze dotrzesz do dedup fleksji. 
# Jeśli za każdym razem union równa się największemu z pojedynczych zbiorów (czyli jeden wariant zawierał już wszystko),
# to merge nie dodał nic poza samym ujednoliceniem nazwy — wciąż wartościowe (jeden wpis zamiast dwóch konkurujących), ale nie wzbogaca puli tłumaczeń.

# To daje odpowiedź na Twoje pytanie bez drugiego przebiegu pipeline'u, bez _row_id, i w czasie proporcjonalnym do rzeczywistej skali problemu (kilkadziesiąt grup, nie tysiące wierszy).

In [ ]:
# Duplikaty w samej all_translations_dedup między różnymi targetami tego samego eng_lemma
# sprawdź, czy pol_to_eng rzeczywiście jest injekcją (len(pol_to_eng) == len(set(pol_to_eng.values()) mapped uniquely)),
# bo to fundament poprawności całej substytucji w DOM

assert len(pol_to_eng) == len({(k) for k in pol_to_eng})  # trivially true — real check below
from collections import Counter
rev = Counter(pol_to_eng.values())
print("Targets winning >1 polish word:", sum(1 for v in rev.values() if v > 0))  # expected — many words map to one target, that's fine
# Real invariant to check: no polish word maps to two different targets
assert len(pol_to_eng) == len(set(pol_to_eng.keys()))

In [ ]:
df_nophrase = df_sorted[df_sorted['eng_word_type']!='phrase']
n_total = len(df_nophrase)
n_fallback = (df_nophrase["_freq_rank"] == 999_999).sum()
print(f"Words WITHOUT frequency rank (fallback): {n_fallback}/{n_total} ({n_fallback/n_total:.1%})")
print(df_nophrase.loc[df_nophrase["_freq_rank"]==999_999, "target"].sample(min(30,n_fallback)).tolist())

In [ ]:
# Rozkład liczby form na target (histogram) pomoże wychwycić targety, 
# które "wygrały" absurdalnie dużo form kosztem innych — potencjalny sygnał złego _priority/_criticality tie-breakingu.

# Sprawdź recall na losowej próbce ręcznie — 
# wybierz 20 losowych target i zweryfikuj czy all_translations_dedup faktycznie zawiera sensowne, kompletne odmiany 
# (np. czy zamek ma zamku, zamkiem itd., a nie tylko mianownik)

In [ ]:
df_sorted["t_word_type"] = df_sorted["all_translations_dedup"].apply(lambda s: [classify_token_word_type(tags) for tags in analyse_translations(" ,".join(s))])
df_sorted["t_word_type"] = df_sorted["t_word_type"].apply(summerise_translations_word_type)
single_uninflected = df_sorted[
    (df_sorted["all_translations_dedup"].apply(len) == 1)
    & (df_sorted["t_word_type"] != "phrase")
    & (df_sorted["eng_word_type"] != "phrase")
]
print(f"Suspicious single-translation, zero-inflection targets: {len(single_uninflected)}")

pd.reset_option('display.max_colwidth')
single_uninflected[['target', 'duolingo_translations', 'inflection_generation_units', 'generated_translations', 'all_translations', 'all_translations_dedup']]

### Stage 6 - export csv

In [ ]:
df_raw['tran_l']=df_raw['duolingo_translations'].apply(lambda x: [i for i in x.split(',')])
print(len(df_raw))
print(len(df_raw.explode('tran_l')))

In [ ]:
final_table = (
    df_sorted[df_sorted["all_translations_dedup"].apply(len) > 0]
    .explode("all_translations_dedup")
    .rename(columns={"all_translations_dedup": "pol_word"})
    [["pol_word", "target"]]
    .reset_index(drop=True)
)
assert final_table["pol_word"].is_unique  # sanity: injectivity of pol_word → target

In [ ]:
print(len(df_sorted[df_sorted["all_translations_dedup"].apply(len) > 0]))
print(len(final_table))

In [ ]:
final_table = (
    df_sorted[df_sorted["all_translations_dedup"].apply(len) > 0]
    .assign(inflections=lambda d: d["all_translations_dedup"].apply(
        lambda s: ", ".join(sorted(s))
    ))
    .rename(columns={"duolingo_translations": "translations", "all_translations_dedup": "forms", "eng_lexeme_form": "word_type"})
    [["target", "translations", "forms", "word_type"]]
    .reset_index(drop=True)
)

In [ ]:

final_table['word_type'] = final_table['word_type'].apply(lambda x: 'phrase' if x=='OTHER' else x)

In [ ]:
from pathlib import Path
p = Path(INPUT_CSV)
out_path = p.with_name(p.stem + "_with_inflections" + p.suffix)
final_table.to_csv(out_path, index=False, encoding="utf-8-sig", sep=';')
print(f"Saved {len(final_table)} rows to {out_path}")

#### SIDE SCRIPT: empirical tag exploration

In [ ]:
# %% ── SIDE SCRIPT: empirical tag exploration ─────────────────────────────────
# Run morf.generate() on all Polish lemmas from supported_lemmas across final
# rows. Collect all unique tags and their frequencies to inform constraint design.

from collections import Counter

_tag_counter:       Counter = Counter()
_class_tag_counter: dict[str, Counter] = defaultdict(Counter)

_all_pol_lemmas: set[str] = set()
for idx in df[final].index:
    sup = df.at[idx, 'supported_lemmas']
    if not sup:
        sup = df.at[idx, 'supported_lemmas_analysed']
    if not sup:
        continue
    for entry in sup:
        _all_pol_lemmas.add(entry['lemma'])


for pol_lemma in sorted(_all_pol_lemmas):
    try:
        gen_results = morf.generate(pol_lemma)
    except Exception as e:
        print(f"  generate error for '{pol_lemma}': {e}")
        continue
    for entry in gen_results:
        form, lemma_out, tag, *_ = entry
        if tag == 'ign':
            continue
        parsed = parse_morf_tag(tag)
        # if has_tag_value(parsed, 'num'):
        #     print(form, lemma_out, tag)
        mc = tag_class(parsed)
        _tag_counter[tag] += 1
        _class_tag_counter[mc][tag] += 1

# ── Output 1: all unique tags sorted by class then frequency ──────────────────
print([mc for mc in sorted(_class_tag_counter)])


## OLD METHOD

In [ ]:
OUTPUT_CSV = Path(INPUT_CSV).with_name(f"{Path(INPUT_CSV).stem}_morphological_expansion_old.csv").as_posix()

df = pd.read_csv(INPUT_CSV, sep=";", names=["target_word", "translations"], header=0, dtype=str)
df["target_word"]= df["target_word"].str.strip()
df["translations"] = df["translations"].str.strip()

print(f"Loaded {len(df)} rows.")
print(df.head())

### Helpers

Morfeusz2 analysis returns a list of (start, end, (orth, lemma, tag, features, flags)) tuples.
For a phrase like "pies" you get one or more interpretations with different lemmas.  
The lemma is the base/dictionary form of the word.  
 
Example:  
>morf.analyse("psy") → 
```
[
    (0,1,('psy','pies:Sm1','depr:pl:nom.acc.voc:m2',['nazwa_pospolita'],['pot.'])),
    (0,1,('psy','pies:Sm2','subst:pl:nom.acc.voc:m2',['nazwa_pospolita'],[]))
]
```
>meaning "psy" is an inflected form of lemma "pies"


In [ ]:
# =============================================================================
# CELL 3: Morfeusz helpers
# =============================================================================

def morf_analyse(word: str) -> pd.DataFrame:
    data = [(r, c, *inner) for r, c, inner in morf.analyse(word.lower())]
    return pd.DataFrame(
        data, columns=["row", "col", "surface", "lemma", "tag", "commonness", "qualifiers"]
    )


def morf_generate(lemma: str) -> pd.DataFrame:
    data = [entry for entry in morf.generate(lemma)]
    return pd.DataFrame(data, columns=["word", "lemma", "tag", "features", "flags"])


def get_lemmas_with_tags(word: str) -> dict[str, str]:
    """
    Returns {lemma: tag} where tag is the morphological tag of THIS word form.
    This constrains which generated forms are valid for the lemma.
    e.g. lepszy → {dobry:A: "adj:sg:nom:m1:com"} → generate only com forms
         dobry  → {dobry:A: "adj:sg:nom:m1:pos"} → generate only pos forms
         pisać  → {pisać:   "inf:imperf"}         → generate only imperf verb forms
    For phrases: returns {phrase: "phrase"} as sentinel.
    """
    if len(word.strip().split()) > 1:
        return {word.strip().lower(): "phrase"}
    df = morf_analyse(word)
    if df.empty:
        return {}
    result = {}
    for _, row in df.iterrows():
        lemma = row["lemma"]
        if lemma not in result:
            result[lemma] = row["tag"]
    return result


# Smoke test
print(morf_analyse("lepszy"))
print(get_lemmas_with_tags("lepszy"))
print(get_lemmas_with_tags("oglądać"))

### Build lemma_candidates per vocab row


Translations cell may be a single word, a phrase, or a comma-separated list.
We analyse every token and collect the union of all candidate lemmas.

In [ ]:
# =============================================================================
# CELL 4: Build lemma_candidates with provenance
# =============================================================================

def build_lemma_candidates(translations_str: str) -> tuple[dict[str, str], dict[str, str]]:
    """
    Returns (direct, derived) where each is {lemma: source_form_tag}.
    direct:  lemma base form matches the translation token exactly
    derived: lemma arrived via morphological analysis of an inflected form
    The source_form_tag (tag of the translation token) constrains form generation.
    """
    direct:  dict[str, str] = {}
    derived: dict[str, str] = {}

    for translation in translations_str.split(","):
        translation_clean = translation.strip().lower()
        if not translation_clean:
            continue
        lemmas_with_tags = get_lemmas_with_tags(translation_clean)
        for lemma, tag in lemmas_with_tags.items():
            if lemma.split(":")[0].lower() == translation_clean:
                direct[lemma] = tag
            else:
                derived[lemma] = tag

    # direct claim overrides derived for same lemma
    for lemma in direct:
        derived.pop(lemma, None)

    return direct, derived


df[["direct_lemmas", "derived_lemmas"]] = df["translations"].apply(
    lambda t: pd.Series(build_lemma_candidates(t))
)
df["lemma_candidates"] = df.apply(
    lambda row: {**row["direct_lemmas"], **row["derived_lemmas"]}, axis=1
)

print("Sample with provenance:")
print(df[["target_word", "direct_lemmas", "derived_lemmas"]].head(10).to_string())

### Pre-heuristic stats

In [ ]:
# =============================================================================
# CELL 4b: Pre-heuristic stats
# =============================================================================

target_dupes = df[df.duplicated("target_word", keep=False)]["target_word"].unique()
print(f"=== Target word duplicates: {len(target_dupes)} ===")
if len(target_dupes):
    print(df[df["target_word"].isin(target_dupes)][["target_word", "translations"]].to_string())

all_lemmas_flat = [l for cands in df["lemma_candidates"] for l in cands]
total_lemmas  = len(all_lemmas_flat)
unique_lemmas = len(set(all_lemmas_flat))
print(f"\n=== Lemma pool ===")
print(f"Total lemma slots : {total_lemmas}")
print(f"Unique lemmas     : {unique_lemmas}")
print(f"Contested slots   : {total_lemmas - unique_lemmas}")

from collections import Counter
lemma_counts = Counter(all_lemmas_flat)
contested = {l: c for l, c in lemma_counts.items() if c > 1}
print(f"\n=== Contested lemmas: {len(contested)} ===")
if contested:
    print(
        pd.DataFrame.from_dict(contested, orient="index", columns=["row_count"])
        .sort_values("row_count", ascending=False)
        .head(20)
        .to_string()
    )

contested_set = set(contested)
df["_has_contested"] = df["lemma_candidates"].apply(lambda d: bool(set(d) & contested_set))
print(f"\n=== Rows with contested lemma: {df['_has_contested'].sum()} / {len(df)} ===")
print(df[df["_has_contested"]][["target_word", "translations", "lemma_candidates"]].head(10).to_string())
print(f"\n=== Candidate set size distribution ===")
print(df["lemma_candidates"].apply(len).describe())
df.drop(columns=["_has_contested"], inplace=True)

In [ ]:
mask = df["lemma_candidates"].map(lambda s: "zestresować" in s)
df[mask]

In [ ]:
# =============================================================================
# CELL 4c: Translation-level stats
# =============================================================================
from collections import Counter

def parse_translations(t: str) -> list[str]:
    return [x.strip().lower() for x in t.split(",") if x.strip()]

df["_tlist"] = df["translations"].apply(parse_translations)
all_trans = [t for tl in df["_tlist"] for t in tl]
trans_counts = Counter(all_trans)
shared = {t: c for t, c in trans_counts.items() if c > 1}

print(f"=== Translations in 2+ rows: {len(shared)} ===")
if shared:
    print(pd.DataFrame.from_dict(shared, orient="index", columns=["row_count"])
          .sort_values("row_count", ascending=False).head(20).to_string())

shared_set = set(shared)
df["_has_shared"] = df["_tlist"].apply(lambda tl: bool(set(tl) & shared_set))
print(f"\n=== Rows with shared translation: {df['_has_shared'].sum()} / {len(df)} ===")
print(df[df["_has_shared"]][["target_word", "translations"]].head(10).to_string())

df["_tcount"] = df["_tlist"].apply(len)
print(f"\n=== Translations per row ===")
print(df["_tcount"].describe())
print(f"1 translation : {(df['_tcount']==1).sum()}")
print(f"3+ translations: {(df['_tcount']>=3).sum()}")
print("\n=== Top 10 by translation count ===")
print(df.nlargest(10, "_tcount")[["target_word", "translations", "_tcount"]].to_string())
df.drop(columns=["_tlist", "_has_shared", "_tcount"], inplace=True)

### Cell 5: "Smaller Set Wins" with provenance priority

Lemma assignment heuristic — "smaller set wins with provenance priority"

Each Polish lemma candidate must be assigned to exactly one English target word (one-to-one constraint). When a lemma appears as a candidate in multiple rows, it is assigned to the row that has the strongest claim to it and removed from all others.

Claim strength is evaluated by three criteria in order:

1. Provenance — if the lemma's base form appears verbatim as a translation token in a row, that row has a direct claim. Rows where the lemma arrived only as a morphological byproduct of an inflected translation have a derived claim. Direct beats derived.

2. Set size — among rows with equal provenance, the row with fewer total lemma candidates wins. Fewer candidates indicates less morphological noise and a more focused translation set, making the claim more specific.

3. Index — deterministic tiebreaker. Lower row index wins.

Set sizes are evaluated against the original pre-heuristic state so that earlier resolutions do not influence later ones.

Rows that lose all their lemma candidates are logged as coverage warnings — they are semantically subsumed by other rows and will generate no surface forms.

In [ ]:
# =============================================================================
# CELL 5: Heuristic — assign contested lemmas to most confident row
# =============================================================================
#
# For each lemma appearing in 2+ rows, assign exclusively to the winner:
#   1. Provenance  — direct (translation token IS the lemma base) beats derived
#   2. Set size    — fewer candidates = more focused row (snapshotted pre-loop)
#   3. Index       — deterministic last resort
#
# Sizes snapshotted before loop so earlier resolutions don't affect later ones.

def apply_smaller_set_wins(df: pd.DataFrame) -> pd.DataFrame:
    original_sizes = {idx: len(row["lemma_candidates"]) for idx, row in df.iterrows()}

    lemma_to_rows: dict[str, list[int]] = defaultdict(list)
    for idx, row in df.iterrows():
        for lemma in row["lemma_candidates"]:
            lemma_to_rows[lemma].append(idx)

    for lemma, row_indices in lemma_to_rows.items():
        if len(row_indices) < 2:
            continue

        def sort_key(i, lemma=lemma):
            provenance = 0 if lemma in df.at[i, "direct_lemmas"] else 1
            return (provenance, original_sizes[i], i)

        row_indices_sorted = sorted(row_indices, key=sort_key)
        winner_idx = row_indices_sorted[0]

        logger.info(
            f"Contested '{lemma}' → '{df.at[winner_idx, 'target_word']}' "
            f"(provenance={'direct' if lemma in df.at[winner_idx, 'direct_lemmas'] else 'derived'}, "
            f"size={original_sizes[winner_idx]}), removed from: "
            + ", ".join(
                f"'{df.at[i, 'target_word']}'("
                f"{'direct' if lemma in df.at[i, 'direct_lemmas'] else 'derived'}, "
                f"size={original_sizes[i]})"
                for i in row_indices_sorted[1:]
            )
        )

        for idx in row_indices_sorted[1:]:
            df.at[idx, "lemma_candidates"] = {k: v for k, v in df.at[idx, "lemma_candidates"].items() if k != lemma}
            df.at[idx, "direct_lemmas"]    = {k: v for k, v in df.at[idx, "direct_lemmas"].items()    if k != lemma}
            df.at[idx, "derived_lemmas"]   = {k: v for k, v in df.at[idx, "derived_lemmas"].items()   if k != lemma}

    empty_rows = df[df["lemma_candidates"].apply(len) == 0]
    if not empty_rows.empty:
        logger.warning(f"{len(empty_rows)} rows have zero lemmas after heuristic:")
        print(empty_rows[["target_word", "translations"]].to_string())

    return df


df = apply_smaller_set_wins(df.copy())

### Post heuristic

In [ ]:
# =============================================================================
# CELL 6: Post-heuristic audit
# =============================================================================

df["lemma_candidates"] = df["lemma_candidates"].apply(lambda d: dict(d))

print("\n=== Post-heuristic inter-row duplicate audit ===")
lemma_row_map: dict[str, list] = defaultdict(list)
for idx, row in df.iterrows():
    for lemma in row["lemma_candidates"]:
        lemma_row_map[lemma].append((idx, row["target_word"]))

duplicates_found = False
for lemma, appearances in lemma_row_map.items():
    if len(appearances) > 1:
        duplicates_found = True
        logger.warning(
            f"Lemma '{lemma}' still in multiple rows: "
            + ", ".join(f"row {i} ('{w}')" for i, w in appearances)
        )
if not duplicates_found:
    print("No cross-row duplicate lemmas — clean.")

print("\n=== Post-heuristic set size distribution ===")
print(df["lemma_candidates"].apply(len).describe())

zero_rows = df[df["lemma_candidates"].apply(len) == 0]
print(f"\nRows with zero lemmas: {len(zero_rows)}")
if not zero_rows.empty:
    print(zero_rows[["target_word", "translations"]].to_string())

In [ ]:
# =============================================================================
# CELL 6b: Zero-lemma row classification
# =============================================================================

ENGLISH_FUNCTION_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "not", "no", "nor",
    "and", "or", "but", "if", "in", "on", "at", "to", "for", "of",
    "from", "with", "by", "as", "it", "its", "he", "she", "they",
    "we", "i", "you", "him", "her", "us", "them", "his", "my", "our",
    "their", "this", "that", "these", "those", "there", "here", "some",
    "any", "who", "what", "which", "how", "when", "where", "why",
    "don't", "isn't", "it's", "there's", "who is", "is it", "p.m.", "a.m."
}

zero_mask = df["lemma_candidates"].apply(len) == 0
zero_df = df[zero_mask].copy()
zero_df["zero_category"] = zero_df["target_word"].apply(
    lambda w: "function_word" if w.lower().strip() in ENGLISH_FUNCTION_WORDS else "content_word"
)
print(f"Zero-lemma rows by category:")
print(zero_df["zero_category"].value_counts())
print("\nContent words with zero lemmas (need attention):")
print(zero_df[zero_df["zero_category"] == "content_word"][["target_word", "translations"]].to_string())

In [ ]:
# =============================================================================
# CELL 7: Finalize selected_lemmas
# =============================================================================

df["selected_lemmas"] = df["lemma_candidates"]  # dict[lemma → source_form_tag]

print("Selected lemmas sample:")
print(df[["target_word", "selected_lemmas"]].head(10).to_string())

### Form Generation

In [ ]:
# =============================================================================
# CELL 8: Tag-aware form generation
# =============================================================================

ALWAYS_EXCLUDE_SEGMENTS = {":neg", ":agl"}

# Degree-constrained: only generate forms matching source degree
DEGREE_TAGS = {"pos", "com", "sup"}

# Verb surface classes allowed from verb lemmas
VERB_ALLOWED   = {"inf", "fin", "praet", "impt", "winien", "bedzie", "pred", "fut"}
VERB_EXCLUDED  = {"ger", "pcon", "pant", "pact", "ppas", "aglt", "imps"}

ADJ_CLASSES = {"adj", "adja", "adjp", "adjc"}


def get_degree(tag: str) -> str | None:
    for seg in tag.split(":"):
        if seg in DEGREE_TAGS:
            return seg
    return None


def get_aspect(tag: str) -> str | None:
    for seg in tag.split(":"):
        if seg in ("perf", "imperf"):
            return seg
    return None


def should_keep_form(generated_tag: str, source_form_tag: str) -> bool:
    """
    Decides whether a generated form is within the same English-word boundary
    as the source translation form.
    """
    if source_form_tag in ("phrase", "ign"):
        return True

    # Never generate negated or agglutinative forms
    if any(seg in generated_tag for seg in ALWAYS_EXCLUDE_SEGMENTS):
        return False

    source_class    = source_form_tag.split(":")[0]
    generated_class = generated_tag.split(":")[0]

    # --- Adjectives / adverbs: constrain by degree ---
    if source_class in ADJ_CLASSES or source_class == "adv":
        source_degree    = get_degree(source_form_tag)
        generated_degree = get_degree(generated_tag)
        # degree must match exactly — good/better/best are different English words
        if source_degree != generated_degree:
            return False
        # stay within adjectival/adverbial surface classes
        if generated_class not in ADJ_CLASSES | {"adv"}:
            return False
        return True

    # --- Verbs: constrain by aspect, exclude derived categories ---
    if source_class in VERB_ALLOWED:
        if generated_class in VERB_EXCLUDED:
            return False
        if generated_class not in VERB_ALLOWED:
            return False
        # aspect must match — perf and imperf are different English words
        source_aspect    = get_aspect(source_form_tag)
        generated_aspect = get_aspect(generated_tag)
        if source_aspect is not None and generated_aspect is not None:
            if source_aspect != generated_aspect:
                return False
        return True

    # --- Participial / gerundial lemmas: only if source was already that class ---
    if source_class in VERB_EXCLUDED:
        return generated_class == source_class

    # --- Nouns, pronouns, numerals, uninflected: allow all forms of same base class ---
    return generated_class == source_class


def get_forms_for_lemma(lemma: str, source_form_tag: str) -> set[str]:
    # Phrases: return verbatim — morf.generate() only accepts single tokens
    if source_form_tag == "phrase" or len(lemma.strip().split()) > 1:
        return {lemma.strip().lower()}

    df_forms = morf_generate(lemma)

    if df_forms.empty:
        logger.warning(f"morf_generate empty for '{lemma}' — using literal")
        return {lemma.lower()}

    if (df_forms["tag"] == "ign").all():
        logger.warning(f"Lemma '{lemma}' unrecognised (ign) — using literal")
        return {lemma.lower()}

    filtered = df_forms[df_forms["tag"].apply(lambda t: should_keep_form(t, source_form_tag))]

    if filtered.empty:
        logger.warning(f"All forms filtered for '{lemma}' (source_tag={source_form_tag}) — falling back to unfiltered")
        return set(df_forms["word"].str.lower())

    return set(filtered["word"].str.lower())


def generate_forms_for_row(lemma_dict: dict[str, str]) -> str:
    all_forms: set[str] = set()
    for lemma, source_form_tag in lemma_dict.items():
        all_forms |= get_forms_for_lemma(lemma, source_form_tag)
    return ",".join(sorted(all_forms))


df["forms"] = df["selected_lemmas"].apply(generate_forms_for_row)

print("Sample output with forms:")
print(df[["target_word", "translations", "selected_lemmas", "forms"]].head(10).to_string())

### Export CSV

In [ ]:
# Reload helper — paste this at top of any future investigation notebook
def deserialize_lemma_dict(s: str) -> dict[str, str]:
    if not s or pd.isna(s):
        return {}
    result = {}
    for entry in s.split("|"):
        parts = entry.rsplit("/", 1)  # rsplit to handle lemmas that might contain /
        if len(parts) == 2:
            result[parts[0]] = parts[1]
    return result

In [ ]:
# =============================================================================
# CELL 9: Export CSV
# =============================================================================
# Keeps all intermediate columns for debugging and reanalysis without rerunning.
# Columns:
#   target_word      — English target
#   translations     — raw Duolingo translation string
#   source           — vocab source (duolingo/anki/etc)
#   direct_lemmas    — {lemma: source_form_tag} — translation token matched lemma base exactly
#   derived_lemmas   — {lemma: source_form_tag} — arrived via morphological analysis
#   lemma_candidates — {lemma: source_form_tag} — union of direct + derived post-heuristic
#   selected_lemmas  — same as lemma_candidates (explicit checkpoint column)
#   forms            — all generated Polish surface forms, comma-separated

def serialize_lemma_dict(d: dict[str, str]) -> str:
    """Serialize {lemma: tag} as 'lemma/tag' entries pipe-separated."""
    return "|".join(f"{lemma}/{tag}" for lemma, tag in d.items())


# output_df = df[[
#     "target_word",
#     "translations",
#     "source",
#     "direct_lemmas",
#     "derived_lemmas",
#     "lemma_candidates",
#     "selected_lemmas",
#     "forms",
# ]].copy()

# output_df["direct_lemmas"]    = output_df["direct_lemmas"].apply(serialize_lemma_dict)
# output_df["derived_lemmas"]   = output_df["derived_lemmas"].apply(serialize_lemma_dict)
# output_df["lemma_candidates"] = output_df["lemma_candidates"].apply(serialize_lemma_dict)
# output_df["selected_lemmas"]  = output_df["selected_lemmas"].apply(serialize_lemma_dict)

output_df = df[[
    "target_word",
    "translations",
    "forms",
]].copy()

output_df.to_csv(OUTPUT_CSV, index=False, sep=";")
print(f"Exported {len(output_df)} rows to '{OUTPUT_CSV}'")
print(output_df.head(5).to_string())

In [ ]:
# =============================================================================
# CELL 10: Pipeline summary statistics
# =============================================================================

df_original = pd.read_csv(INPUT_CSV, sep=";", names=["target_word", "translations", "source"], header=0, dtype=str)
trans_per_row     = df_original["translations"].apply(lambda t: len([x for x in t.split(",") if x.strip()]))
total_trans_start = trans_per_row.sum()
rows_start        = len(df_original)

rows_end      = len(output_df[output_df["forms"].str.len() > 0])
forms_per_row = output_df["forms"].apply(lambda f: len(f.split(",")) if f else 0)
total_forms_end = forms_per_row.sum()

print("=" * 50)
print("PIPELINE SUMMARY")
print("=" * 50)
print(f"Rows at start                : {rows_start}")
print(f"Rows with forms at end       : {rows_end}")
print(f"Rows lost (zero forms)       : {rows_start - rows_end}")
print()
print(f"Total translations (input)   : {total_trans_start}")
print(f"Total surface forms (output) : {total_forms_end}")
print()
print(f"Avg translations per row     : {trans_per_row.mean():.2f}")
print(f"Avg forms per row            : {forms_per_row[forms_per_row > 0].mean():.2f}")
print(f"Avg forms per row (all rows) : {forms_per_row.mean():.2f}")
print()
print(f"Forms expansion ratio        : {total_forms_end / total_trans_start:.1f}x")
print("=" * 50)

print("\nForms per row distribution:")
print(forms_per_row[forms_per_row > 0].describe().round(2))

## compare

In [ ]:
NEW_FILE = Path(r"C:\Users\rrkar\Downloads\duolingo_vocab_tata_16052026_2026-05-19_2130_with_inflections.csv")  # kolumny: target, duolingo_translations, inflections
OLD_FILE = Path(r"C:\Users\rrkar\Downloads\duolingo_vocab_tata_16052026_2026-05-19_2130_morphological_expansion_old.csv")   # kolumny: target_word, translations, forms 
# NEW_FILE = Path(r'C:\Users\rrkar\Downloads\duolingo_vocab_mama_16052026_2026-05-19_2131_with_inflections.csv')
# OLD_FILE = Path(r'C:\Users\rrkar\Downloads\duolingo_vocab_mama_16052026_2026-05-19_2131_morphological_expansion_old.csv')

In [ ]:
import pandas as pd


def to_set(s):
    if pd.isna(s) or not str(s).strip():
        return set()
    return set(x.strip() for x in str(s).split(",") if x.strip())

df_old = pd.read_csv(OLD_FILE, sep=';')
df_new = pd.read_csv(NEW_FILE, sep=';')

df_old = df_old.rename(columns={
    "target_word": "target",
    "translations": "translations_old",
    "forms": "forms_old",
})
df_new = df_new.rename(columns={
    "duolingo_translations": "translations_new",
    "inflections": "inflections_new",
})

df_old["target"] = df_old["target"].astype(str).str.strip()
df_new["target"] = df_new["target"].astype(str).str.strip()

# --- 1. Target word coverage ---
old_targets = set(df_old["target"])
new_targets = set(df_new["target"])

only_old = old_targets - new_targets
only_new = new_targets - old_targets
common = old_targets & new_targets

print("=== TARGET COVERAGE ===")
print(f"Old file targets: {len(old_targets)}")
print(f"New file targets: {len(new_targets)}")
print(f"Common targets:   {len(common)}")
print(f"Only in OLD (lost in new): {len(only_old)}")
print(f"Only in NEW (gained):      {len(only_new)}")
print()
print("Sample only in OLD:", list(only_old)[:20])
print("Sample only in NEW:", list(only_new)[:20])

pd.DataFrame({"target": sorted(only_old)}).to_csv("targets_only_in_old.csv", index=False)
pd.DataFrame({"target": sorted(only_new)}).to_csv("targets_only_in_new.csv", index=False)

# --- 2. Inflection diff on common targets ---
df_old_idx = df_old.set_index("target")
df_new_idx = df_new.set_index("target")

rows = []
for t in common:
    old_val = df_old_idx.loc[t, "forms_old"]
    new_val = df_new_idx.loc[t, "inflections_new"]

    # guard against duplicate target rows (returns a Series instead of scalar)
    if isinstance(old_val, pd.Series):
        old_forms = set()
        for v in old_val:
            old_forms |= to_set(v)
    else:
        old_forms = to_set(old_val)

    if isinstance(new_val, pd.Series):
        new_forms = set()
        for v in new_val:
            new_forms |= to_set(v)
    else:
        new_forms = to_set(new_val)

    lost = old_forms - new_forms
    gained = new_forms - old_forms
    if lost or gained:
        rows.append({
            "target": t,
            "old_forms": sorted(old_forms),
            "new_forms": sorted(new_forms),
            "lost_forms":sorted(lost),
            "gained_forms": sorted(gained),
            "n_lost": len(lost),
            "n_gained": len(gained),
        })

diff_df = pd.DataFrame(rows)
if len(diff_df):
    diff_df = diff_df.sort_values("n_lost", ascending=False)
diff_df.to_csv("inflections_diff.csv", index=False)

n_identical = len(common) - len(diff_df)
print()
print("=== INFLECTIONS DIFF (on common targets) ===")
print(f"Identical inflections: {n_identical}/{len(common)}")
print(f"Targets with differences: {len(diff_df)}")
print(f"Total forms lost:   {diff_df['n_lost'].sum() if len(diff_df) else 0}")
print(f"Total forms gained: {diff_df['n_gained'].sum() if len(diff_df) else 0}")
print()
print("Top 15 targets with most lost forms:")
if len(diff_df):
    print(diff_df.head(15)[["target","gained_forms","lost_forms"]].to_string(index=False))
else:
    print("(no differences found)")

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

pd.reset_option('display.max_rows')
# pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')

diff_df[diff_df['n_gained']>0]#[["target","gained_forms"]]
diff_df.sort_values('n_lost', ascending=False).head(10)